# Entity Resolution — GC1 Temporal Knowledge Graph

This notebook resolves source-scoped entity mentions produced by `Entity_Extraction.ipynb` into canonical project-instance entities, without OpenAI/LLM calls.

**Pipeline:** validate extraction contract → flatten mentions → normalize → deterministic ID matching → normalized exact matching → type-aware candidates → MiniLM semantic similarity → contextual validation → conservative clustering → canonical IDs → rewrite extracted structures → integrity checks → optional pairwise ER evaluation → database-neutral + Neo4j-ready exports.

### Resolution policy
- Resolution is **instance-scoped**: entities from different `instance_id`s are never merged.
- Entity types must match by default.
- Explicit identifiers are resolved deterministically.
- Ambiguous pairs are **not** automatically merged.
- The original extraction JSONL files are never modified.

In [2]:
# ============================================================
# STEP 1: INSTALL DEPENDENCIES (Google Colab)
# ============================================================
!pip -q install sentence-transformers scikit-learn pandas numpy tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# ============================================================
# STEP 2: IMPORTS + CONFIGURATION
# ============================================================
from pathlib import Path
from collections import defaultdict, Counter
import copy, hashlib, json, math, re, unicodedata

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support

# CHANGE THIS to the common folder containing the 30 extraction-output folders.
# The notebook recursively discovers the three filenames below.
EXTRACTION_ROOT = Path('/content/drive/MyDrive/GC1')
OUTPUT_ROOT = EXTRACTION_ROOT / 'entity_resolution_output'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

JIRA_OUTPUT_NAME = 'jira_extracted.jsonl'
PRD_OUTPUT_NAME = 'prd_extracted.jsonl'
MEETING_OUTPUT_NAME = 'meetings_extracted.jsonl'
EXPECTED_FILENAMES = {JIRA_OUTPUT_NAME, PRD_OUTPUT_NAME, MEETING_OUTPUT_NAME}

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Conservative defaults. Tune on labelled ER pairs; do not report these as optimal
# until the evaluation section has been run on your own validation labels.
SEMANTIC_ACCEPT_THRESHOLD = 0.88
AMBIGUOUS_REVIEW_THRESHOLD = 0.78
CONTEXT_WEIGHT = 0.20
SEMANTIC_WEIGHT = 0.80
COMBINED_ACCEPT_THRESHOLD = 0.88

# Keep matching type-strict initially. Add explicit compatible pairs only if justified.
TYPE_COMPATIBILITY = {}

print('Input root :', EXTRACTION_ROOT)
print('Output root:', OUTPUT_ROOT)

Input root : /content/drive/MyDrive/GC1
Output root: /content/drive/MyDrive/GC1/entity_resolution_output


In [2]:
# ============================================================
# STEP 3: DISCOVER ALL FINAL EXTRACTION JSONL FILES
# ============================================================
def discover_extraction_files(root: Path):
    files = []
    for name in sorted(EXPECTED_FILENAMES):
        files.extend(root.rglob(name))
    return sorted(set(files))

extraction_files = discover_extraction_files(EXTRACTION_ROOT)
print('Discovered extraction files:', len(extraction_files))
for p in extraction_files[:15]:
    print(' -', p)
if len(extraction_files) > 15:
    print(' ...')

if not extraction_files:
    raise FileNotFoundError(
        'No final extraction JSONL files found. Set EXTRACTION_ROOT to the folder '
        'containing the 30 project-instance extraction outputs.'
    )

Discovered extraction files: 90
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/01_JBEAP/JBEAP-I01/jira_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/01_JBEAP/JBEAP-I01/meetings_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/01_JBEAP/JBEAP-I01/prd_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/02_JBIDE/JBIDE-I01/jira_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/02_JBIDE/JBIDE-I01/meetings_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/02_JBIDE/JBIDE-I01/prd_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/03_SAK/SAK-I01/jira_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/03_SAK/SAK-I01/meetings_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/03_SAK/SAK-I01/prd_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/04_JBDS/JBDS-I01/jira_extracted.jsonl
 - /content/drive/MyDrive/GC1/GC1_Entity_Extraction/0

In [3]:
# ============================================================
# STEP 4: LOAD + VALIDATE THE EXTRACTION CONTRACT
# ============================================================
REQUIRED_TOP_LEVEL = {
    'repository_project', 'project_key', 'instance_id', 'source',
    'entities', 'relationships', 'events', 'decisions', 'action_items',
    'blockers', 'dependencies', 'temporal', 'attributes', 'provenance'
}
REQUIRED_SOURCE = {'source_type', 'source_id', 'source_timestamp'}
REQUIRED_ENTITY = {
    'local_id', 'type', 'name', 'normalized_name', 'attributes',
    'extraction_method', 'confidence'
}

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                obj = json.loads(line)
                obj['_input_file'] = str(path)
                obj['_input_line'] = line_no
                rows.append(obj)
    return rows

def validate_record(record):
    errors = []
    missing = REQUIRED_TOP_LEVEL - set(record)
    if missing: errors.append(f'missing top-level: {sorted(missing)}')
    if isinstance(record.get('source'), dict):
        m = REQUIRED_SOURCE - set(record['source'])
        if m: errors.append(f'missing source fields: {sorted(m)}')
    else:
        errors.append('source is not an object')
    if not isinstance(record.get('entities'), list):
        errors.append('entities is not a list')
    else:
        for i, e in enumerate(record['entities']):
            m = REQUIRED_ENTITY - set(e)
            if m: errors.append(f'entity[{i}] missing: {sorted(m)}')
    return errors

all_records, contract_errors = [], []
for path in tqdm(extraction_files, desc='Loading JSONL'):
    for record in load_jsonl(path):
        errs = validate_record(record)
        if errs:
            contract_errors.append({
                'file': record['_input_file'], 'line': record['_input_line'],
                'errors': errs
            })
        all_records.append(record)

print('Records loaded :', len(all_records))
print('Contract errors:', len(contract_errors))
if contract_errors:
    display(pd.DataFrame(contract_errors).head(20))
    raise ValueError('Extraction contract validation failed. Fix errors before resolution.')

instances = sorted({r['instance_id'] for r in all_records})
print('Unique instances:', len(instances))
print(instances[:10])

Loading JSONL:   0%|          | 0/90 [00:00<?, ?it/s]

Records loaded : 5827
Contract errors: 0
Unique instances: 30
['AEROGEAR-I01', 'AMBARI-I01', 'BXMSDOC-I01', 'CONFSERVER-I01', 'DAOS-I01', 'DOCS-I01', 'ENTESB-I01', 'EVG-I01', 'FLEX-I01', 'HBASE-I01']


In [4]:
# ============================================================
# STEP 5: FLATTEN ENTITY MENTIONS
# ============================================================
def safe_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)

mention_rows = []
record_lookup = {}
for record in all_records:
    source = record['source']
    record_key = (record['instance_id'], source['source_type'], source['source_id'])
    record_lookup[record_key] = record
    for entity in record['entities']:
        mention_rows.append({
            'repository_project': record['repository_project'],
            'project_key': record['project_key'],
            'instance_id': record['instance_id'],
            'source_type': source['source_type'],
            'source_id': source['source_id'],
            'source_timestamp': source.get('source_timestamp'),
            'local_id': entity['local_id'],
            'entity_type': str(entity['type']).upper().strip(),
            'name': str(entity['name']).strip(),
            'attributes': entity.get('attributes') or {},
            'attributes_json': safe_json(entity.get('attributes') or {}),
            'extraction_method': entity.get('extraction_method'),
            'extraction_confidence': entity.get('confidence'),
            'input_file': record['_input_file'],
        })

mentions = pd.DataFrame(mention_rows)
if mentions.empty:
    raise ValueError('No entities found in extraction records.')

# local_id is source-scoped. This composite key is globally unique for ER.
mentions['mention_key'] = (
    mentions['instance_id'].astype(str) + '||' +
    mentions['source_type'].astype(str) + '||' +
    mentions['source_id'].astype(str) + '||' +
    mentions['local_id'].astype(str)
)

if mentions['mention_key'].duplicated().any():
    display(mentions[mentions['mention_key'].duplicated(False)].head(20))
    raise ValueError('Duplicate composite mention keys detected.')

print('Entity mentions:', len(mentions))
display(mentions.head())
print('\nEntity types:')
display(mentions['entity_type'].value_counts().rename_axis('type').reset_index(name='count'))

Entity mentions: 61392


,repository_project,project_key,instance_id,source_type,source_id,source_timestamp,local_id,entity_type,name,attributes,attributes_json,extraction_method,extraction_confidence,input_file,mention_key
0,01_JBEAP,JBEAP,JBEAP-I01,JIRA,JBEAP-7698,None,JIRA_JBEAP-7698_E001,PROJECT,JBEAP,"{'project_id': '12313422', 'project_name': 'JB...","{""project_id"": ""12313422"", ""project_name"": ""JB...",structured,1.0,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,JBEAP-I01||JIRA||JBEAP-7698||JIRA_JBEAP-7698_E001
1,01_JBEAP,JBEAP,JBEAP-I01,JIRA,JBEAP-7698,None,JIRA_JBEAP-7698_E002,ISSUE,JBEAP-7698,"{'issue_id': '12665939', 'issue_type': 'Requir...","{""description"": null, ""issue_id"": ""12665939"", ...",structured,1.0,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,JBEAP-I01||JIRA||JBEAP-7698||JIRA_JBEAP-7698_E002
2,01_JBEAP,JBEAP,JBEAP-I01,JIRA,JBEAP-7698,None,JIRA_JBEAP-7698_E003,PERSON,<<|author_displayName|668abc82-6cf3-4b94-9217-...,{},{},structured,1.0,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,JBEAP-I01||JIRA||JBEAP-7698||JIRA_JBEAP-7698_E003
3,01_JBEAP,JBEAP,JBEAP-I01,JIRA,JBEAP-7698,None,JIRA_JBEAP-7698_E004,PERSON,<<|author_displayName|4cba0b88-c353-4847-8bea-...,{},{},structured,1.0,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,JBEAP-I01||JIRA||JBEAP-7698||JIRA_JBEAP-7698_E004
4,01_JBEAP,JBEAP,JBEAP-I01,JIRA,JBEAP-7698,None,JIRA_JBEAP-7698_E005,PERSON,<<|author_displayName|b1413bac-20b5-4688-9f18-...,{},{},structured,1.0,/content/drive/MyDrive/GC1/GC1_Entity_Extracti...,JBEAP-I01||JIRA||JBEAP-7698||JIRA_JBEAP-7698_E005



Entity types:


,type,count
0,ISSUE,37044
1,PERSON,10195
2,PROJECT,5827
3,COMPONENT,5484
4,FEATURE,963
5,SYSTEM,957
6,TEAM,348
7,REQUIREMENT,320
8,MILESTONE,194
9,MEETING,60


In [5]:
# ============================================================
# STEP 6: NORMALIZE ENTITY NAMES + EXTRACT EXPLICIT IDs
# ============================================================
def normalize_name(text):
    text = unicodedata.normalize('NFKC', str(text or '')).casefold().strip()
    text = re.sub(r'[_/]+', ' ', text)
    text = re.sub(r'\s*-\s*', '-', text)  # preserve meaningful ID hyphens
    text = re.sub(r'[^\w\s.-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip(' .')
    return text

# Strong explicit identifiers. Extend patterns only when supported by your data.
ISSUE_ID_RE = re.compile(r'\b[A-Z][A-Z0-9]+-\d+\b', re.I)
REQ_ID_RE = re.compile(r'\bREQ(?:UIREMENT)?[-_ ]?\d+\b', re.I)
MEETING_ID_RE = re.compile(r'\bM\d{3,}\b', re.I)

def explicit_identifier(entity_type, name, attributes):
    et = entity_type.upper()
    raw = str(name or '').strip()
    attrs = attributes or {}
    if et == 'PROJECT':
        return raw.upper() if raw else None
    patterns = []
    if et == 'ISSUE': patterns = [ISSUE_ID_RE]
    elif et == 'REQUIREMENT': patterns = [REQ_ID_RE]
    elif et == 'MEETING': patterns = [MEETING_ID_RE]
    for pat in patterns:
        m = pat.search(raw)
        if m: return re.sub(r'[_ ]+', '-', m.group(0).upper())
    # Also inspect scalar attributes for explicit IDs of the same type.
    for v in attrs.values():
        if isinstance(v, (str, int, float)):
            for pat in patterns:
                m = pat.search(str(v))
                if m: return re.sub(r'[_ ]+', '-', m.group(0).upper())
    return None

mentions['normalized_name'] = mentions['name'].map(normalize_name)
mentions['explicit_id'] = mentions.apply(
    lambda r: explicit_identifier(r['entity_type'], r['name'], r['attributes']), axis=1
)

display(mentions[['instance_id','entity_type','name','normalized_name','explicit_id']].head(20))

,instance_id,entity_type,name,normalized_name,explicit_id
0,JBEAP-I01,PROJECT,JBEAP,jbeap,JBEAP
1,JBEAP-I01,ISSUE,JBEAP-7698,jbeap-7698,JBEAP-7698
2,JBEAP-I01,PERSON,<<|author_displayName|668abc82-6cf3-4b94-9217-...,author displayname 668abc82-6cf3-4b94-9217-472...,None
3,JBEAP-I01,PERSON,<<|author_displayName|4cba0b88-c353-4847-8bea-...,author displayname 4cba0b88-c353-4847-8bea-9c1...,None
4,JBEAP-I01,PERSON,<<|author_displayName|b1413bac-20b5-4688-9f18-...,author displayname b1413bac-20b5-4688-9f18-877...,None
5,JBEAP-I01,COMPONENT,Documentation,documentation,None
6,JBEAP-I01,ISSUE,JBEAP-6030,jbeap-6030,JBEAP-6030
7,JBEAP-I01,ISSUE,JBEAP-8472,jbeap-8472,JBEAP-8472
8,JBEAP-I01,ISSUE,JBEAP-11399,jbeap-11399,JBEAP-11399
9,JBEAP-I01,ISSUE,JBEAP-11410,jbeap-11410,JBEAP-11410


In [6]:
# ============================================================
# STEP 7: BUILD LIGHTWEIGHT CONTEXT FOR EACH MENTION
# ============================================================
# Context comes only from the same extraction record and is used as supporting
# evidence; it does not replace identity matching.

def tokens(text):
    return {t for t in re.findall(r'[a-z0-9]+', normalize_name(text)) if len(t) > 1}

# Neighbor names/types from local relationships.
context_by_key = defaultdict(lambda: {'neighbor_tokens': set(), 'relation_types': set(), 'attribute_tokens': set()})

for record in all_records:
    src = record['source']
    prefix = (record['instance_id'], src['source_type'], src['source_id'])
    local_entities = {e['local_id']: e for e in record['entities']}
    for e in record['entities']:
        key = '||'.join(map(str, (*prefix, e['local_id'])))
        context_by_key[key]['attribute_tokens'] |= tokens(safe_json(e.get('attributes') or {}))
    for rel in record.get('relationships', []):
        a, b = rel.get('source_entity'), rel.get('target_entity')
        rt = str(rel.get('type') or '')
        if a in local_entities and b in local_entities:
            ka = '||'.join(map(str, (*prefix, a)))
            kb = '||'.join(map(str, (*prefix, b)))
            context_by_key[ka]['neighbor_tokens'] |= tokens(local_entities[b].get('name'))
            context_by_key[kb]['neighbor_tokens'] |= tokens(local_entities[a].get('name'))
            context_by_key[ka]['relation_types'].add(rt)
            context_by_key[kb]['relation_types'].add(rt)

def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b: return 0.0
    return len(a & b) / len(a | b)

def context_similarity(key_a, key_b):
    a, b = context_by_key[key_a], context_by_key[key_b]
    neighbor = jaccard(a['neighbor_tokens'], b['neighbor_tokens'])
    relations = jaccard(a['relation_types'], b['relation_types'])
    attrs = jaccard(a['attribute_tokens'], b['attribute_tokens'])
    return 0.50 * neighbor + 0.25 * relations + 0.25 * attrs

print('Context prepared for', len(context_by_key), 'mentions.')

Context prepared for 61392 mentions.


In [7]:
# ============================================================
# STEP 8: UNION-FIND CLUSTER STRUCTURE
# ============================================================
class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
        self.rank = {x: 0 for x in items}
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return False
        if self.rank[ra] < self.rank[rb]: ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]: self.rank[ra] += 1
        return True

uf = UnionFind(mentions['mention_key'])
resolution_edges = []

def accept_pair(a, b, method, score=1.0, semantic=None, context=None):
    uf.union(a, b)
    resolution_edges.append({
        'mention_a': a, 'mention_b': b, 'decision': 'MERGE',
        'method': method, 'score': float(score),
        'semantic_similarity': semantic, 'context_similarity': context
    })

In [8]:
# ============================================================
# STEP 9: DETERMINISTIC EXPLICIT-ID MATCHING
# ============================================================
# Same instance + same type + same explicit ID => same canonical entity.
for (instance_id, etype, explicit_id), group in mentions.dropna(subset=['explicit_id']).groupby(
    ['instance_id','entity_type','explicit_id']
):
    keys = group['mention_key'].tolist()
    for k in keys[1:]:
        accept_pair(keys[0], k, 'EXACT_ID', 1.0)

print('Deterministic merge edges:', sum(e['method']=='EXACT_ID' for e in resolution_edges))

Deterministic merge edges: 10252


In [9]:
# ============================================================
# STEP 10: NORMALIZED EXACT MATCHING
# ============================================================
def safe_for_exact_auto_merge(entity_type, normalized):
    if not normalized: return False
    toks = normalized.split()
    # Avoid automatically merging short PERSON mentions such as "John".
    if entity_type == 'PERSON' and len(toks) < 2:
        return False
    # Avoid extremely short/generic strings for all types.
    if len(normalized) < 3:
        return False
    return True

for (instance_id, etype, norm), group in mentions.groupby(['instance_id','entity_type','normalized_name']):
    if not safe_for_exact_auto_merge(etype, norm):
        continue
    keys = group['mention_key'].tolist()
    for k in keys[1:]:
        accept_pair(keys[0], k, 'NORMALIZED_EXACT', 0.99)

print('Normalized exact merge edges:', sum(e['method']=='NORMALIZED_EXACT' for e in resolution_edges))

Normalized exact merge edges: 20589


In [10]:
# ============================================================
# STEP 11: LOAD OR CREATE MINILM EMBEDDINGS
# ============================================================

import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer

ER_CACHE_DIR = "/content/drive/MyDrive/entity_resolution_cache"
os.makedirs(ER_CACHE_DIR, exist_ok=True)

EMBEDDINGS_PATH = f"{ER_CACHE_DIR}/minilm_embeddings.npy"
MENTION_KEYS_PATH = f"{ER_CACHE_DIR}/minilm_mention_keys.json"

def resolution_text(row):
    ctx = context_by_key[row['mention_key']]

    neighbor_text = ' '.join(sorted(ctx['neighbor_tokens']))
    attr_text = ' '.join(sorted(ctx['attribute_tokens']))

    return (
        f"type: {row['entity_type']} | "
        f"name: {row['name']} | "
        f"attributes: {attr_text[:500]} | "
        f"neighbors: {neighbor_text[:500]}"
    )


mentions['resolution_text'] = mentions.apply(
    resolution_text,
    axis=1
)

current_keys = mentions['mention_key'].tolist()


# ------------------------------------------------------------
# CHECK FOR EXISTING CACHE
# ------------------------------------------------------------

cache_valid = False

if os.path.exists(EMBEDDINGS_PATH) and os.path.exists(MENTION_KEYS_PATH):

    print("Found cached embeddings.")

    with open(MENTION_KEYS_PATH, "r") as f:
        saved_keys = json.load(f)

    if saved_keys == current_keys:
        cache_valid = True
    else:
        print(
            "Cache exists, but mention keys do not match "
            "the current dataset."
        )


# ------------------------------------------------------------
# LOAD OR GENERATE
# ------------------------------------------------------------

if cache_valid:

    print("Loading cached MiniLM embeddings...")

    embeddings = np.load(
        EMBEDDINGS_PATH,
        mmap_mode="r"
    )

else:

    print("Generating MiniLM embeddings...")

    model = SentenceTransformer(MODEL_NAME)

    embeddings = model.encode(
        mentions['resolution_text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # float32 is more than sufficient for cosine similarity
    embeddings = embeddings.astype(np.float32)

    print("Saving embeddings to Google Drive...")

    np.save(
        EMBEDDINGS_PATH,
        embeddings
    )

    with open(MENTION_KEYS_PATH, "w") as f:
        json.dump(current_keys, f)

    # Free the transformer after embedding.
    del model

    import gc
    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Found cached embeddings.
Loading cached MiniLM embeddings...
Embedding shape: (61392, 384)
Embedding dtype: float32


In [11]:
# ============================================================
# STEP 12: CRASH-SAFE TYPE-AWARE CANDIDATE GENERATION
#
# Improvements:
#   1. Per-group Google Drive checkpointing
#   2. Resume after Colab crash
#   3. Stable-ID entity types bypass MiniLM matching
#   4. Block-wise cosine similarity for RAM safety
#   5. Atomic checkpoint writes
#   6. Group-size diagnostics
# ============================================================

import os
import gc
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

# Smaller blocks use less RAM.
BLOCK_SIZE = 256


# ------------------------------------------------------------
# Entity types that should NOT undergo semantic pair matching.
#
# These types have stable identifiers and should already have
# been handled by deterministic/exact resolution.
# ------------------------------------------------------------

SEMANTIC_EXCLUDED_TYPES = {
    "ISSUE",
    "PROJECT",
    "MEETING",
}


# ------------------------------------------------------------
# IMPORTANT:
#
# Add REQUIREMENT here ONLY if your extracted requirements
# consistently carry stable IDs such as REQ-001.
#
# Example:
#
# SEMANTIC_EXCLUDED_TYPES = {
#     "ISSUE",
#     "PROJECT",
#     "MEETING",
#     "REQUIREMENT",
# }
# ------------------------------------------------------------


# ============================================================
# CACHE LOCATIONS
# ============================================================

CANDIDATE_CACHE_DIR = os.path.join(
    ER_CACHE_DIR,
    "candidate_group_cache"
)

os.makedirs(
    CANDIDATE_CACHE_DIR,
    exist_ok=True
)


CANDIDATES_PATH = os.path.join(
    ER_CACHE_DIR,
    "semantic_candidates.parquet"
)


# ============================================================
# OUTPUT COLUMNS
# ============================================================

CANDIDATE_COLUMNS = [
    "instance_id",
    "entity_type",
    "mention_a",
    "mention_b",
    "name_a",
    "name_b",
    "semantic_similarity",
    "context_similarity",
    "combined_score",
]


# ============================================================
# HELPER: EMPTY CANDIDATE DATAFRAME
# ============================================================

def empty_candidate_df():

    return pd.DataFrame(
        columns=CANDIDATE_COLUMNS
    )


# ============================================================
# HELPER: SAFE FILE NAME
# ============================================================

def safe_filename(value):

    value = str(value)

    value = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        value
    )

    return value.strip("_")


# ============================================================
# HELPER: GROUP CHECKPOINT PATH
# ============================================================

def group_checkpoint_path(
    instance_id,
    entity_type
):

    instance_safe = safe_filename(
        instance_id
    )

    type_safe = safe_filename(
        entity_type
    )

    filename = (
        f"{instance_safe}"
        f"__{type_safe}"
        f".parquet"
    )

    return os.path.join(
        CANDIDATE_CACHE_DIR,
        filename
    )


# ============================================================
# HELPER: ATOMIC PARQUET SAVE
# ============================================================

def atomic_parquet_save(
    df,
    path
):
    """
    Crash-safe save.

    Data is first written to a temporary file.

    Only once writing succeeds is the temporary file renamed
    to the final .parquet checkpoint.

    Therefore:

        .parquet = completed group
        .tmp     = interrupted/incomplete group
    """

    temp_path = (
        path + ".tmp"
    )


    # Remove stale temporary file from an earlier crash.
    if os.path.exists(temp_path):

        os.remove(
            temp_path
        )


    df.to_parquet(
        temp_path,
        index=False
    )


    os.replace(
        temp_path,
        path
    )


# ============================================================
# BUILD GROUP LIST
# ============================================================

groups = list(
    mentions.groupby(
        [
            "instance_id",
            "entity_type"
        ],
        sort=True
    )
)


print(
    "Total instance/entity-type groups:",
    len(groups)
)

print(
    "Checkpoint directory:",
    CANDIDATE_CACHE_DIR
)


# ============================================================
# GROUP SIZE DIAGNOSTIC
# ============================================================

group_sizes = pd.DataFrame(
    [
        {
            "instance_id": instance_id,
            "entity_type": etype,
            "mentions": len(group),
            "semantic_matching":
                etype not in SEMANTIC_EXCLUDED_TYPES
        }

        for (
            instance_id,
            etype
        ), group in groups
    ]
)


print()
print(
    "Largest entity groups:"
)

display(
    group_sizes
    .sort_values(
        "mentions",
        ascending=False
    )
    .head(20)
)


# ============================================================
# COUNT COMPLETED CHECKPOINTS
# ============================================================

completed_before_start = 0


for (
    instance_id,
    etype
), group in groups:

    checkpoint_path = (
        group_checkpoint_path(
            instance_id,
            etype
        )
    )


    if os.path.exists(
        checkpoint_path
    ):

        completed_before_start += 1


print()
print(
    "Groups already cached:",
    completed_before_start
)

print(
    "Groups remaining:",
    len(groups)
    - completed_before_start
)

print()


# ============================================================
# PROCESS GROUPS
# ============================================================

for (
    instance_id,
    etype
), group in tqdm(
    groups,
    desc="Candidate groups"
):

    checkpoint_path = (
        group_checkpoint_path(
            instance_id,
            etype
        )
    )


    # ========================================================
    # 1. SKIP COMPLETED GROUPS
    # ========================================================

    if os.path.exists(
        checkpoint_path
    ):

        continue


    # ========================================================
    # 2. SKIP STABLE-ID TYPES
    # ========================================================

    if etype in SEMANTIC_EXCLUDED_TYPES:

        print(
            f"\nSkipping semantic resolution: "
            f"{instance_id} / {etype} "
            f"({len(group):,} mentions)"
        )


        empty_df = (
            empty_candidate_df()
        )


        atomic_parquet_save(
            empty_df,
            checkpoint_path
        )


        print(
            "  ✓ Deterministic-ID type; "
            "semantic checkpoint saved."
        )


        del empty_df

        gc.collect()

        continue


    # ========================================================
    # 3. PREPARE SEMANTIC GROUP
    # ========================================================

    idxs = (
        group.index
        .to_numpy()
    )


    print()

    print(
        f"Processing: "
        f"{instance_id} / {etype} "
        f"({len(idxs):,} mentions)"
    )


    # ========================================================
    # 4. GROUPS WITH < 2 MENTIONS
    # ========================================================

    if len(idxs) < 2:

        empty_df = (
            empty_candidate_df()
        )


        atomic_parquet_save(
            empty_df,
            checkpoint_path
        )


        print(
            "  ✓ Saved empty checkpoint "
            "(fewer than 2 mentions)."
        )


        del empty_df

        gc.collect()

        continue


    # ========================================================
    # 5. GET GROUP EMBEDDINGS
    # ========================================================

    E = np.asarray(
        embeddings[idxs],
        dtype=np.float32
    )


    keys = (
        mentions.loc[
            idxs,
            "mention_key"
        ]
        .tolist()
    )


    names = (
        mentions.loc[
            idxs,
            "name"
        ]
        .tolist()
    )


    n = len(keys)


    # Candidates are retained only for this group.
    group_rows = []


    # ========================================================
    # 6. BLOCK-WISE COSINE SIMILARITY
    # ========================================================

    for start_i in range(
        0,
        n,
        BLOCK_SIZE
    ):

        end_i = min(
            start_i
            + BLOCK_SIZE,
            n
        )


        E_block = E[
            start_i:end_i
        ]


        # ----------------------------------------------------
        # Because Step 11 normalized embeddings:
        #
        # cosine_similarity = dot_product
        #
        # Memory allocation is:
        #
        # BLOCK_SIZE × group_size
        #
        # rather than:
        #
        # group_size × group_size
        # ----------------------------------------------------

        sim_block = (
            E_block @ E.T
        )


        # ====================================================
        # PROCESS EACH ROW IN THE BLOCK
        # ====================================================

        for local_i in range(
            end_i - start_i
        ):

            i = (
                start_i
                + local_i
            )


            # ------------------------------------------------
            # Only inspect upper triangle.
            #
            # If A-B is tested,
            # B-A is not tested again.
            # ------------------------------------------------

            sims = sim_block[
                local_i,
                i + 1:
            ]


            if len(sims) == 0:

                continue


            # =================================================
            # SEMANTIC PRE-FILTER
            # =================================================
            #
            # Filtering occurs BEFORE Python dictionaries are
            # created.
            # =================================================

            candidate_js = np.where(
                sims
                >= AMBIGUOUS_REVIEW_THRESHOLD
            )[0]


            candidate_js = (
                candidate_js
                + i
                + 1
            )


            # =================================================
            # SCORE RETAINED PAIRS
            # =================================================

            for j in candidate_js:

                a = keys[i]
                b = keys[j]


                # --------------------------------------------
                # Already resolved together?
                # --------------------------------------------

                if (
                    uf.find(a)
                    == uf.find(b)
                ):

                    continue


                # --------------------------------------------
                # Semantic similarity
                # --------------------------------------------

                semantic = float(
                    sim_block[
                        local_i,
                        j
                    ]
                )


                # --------------------------------------------
                # Context similarity
                # --------------------------------------------

                ctx = (
                    context_similarity(
                        a,
                        b
                    )
                )


                # --------------------------------------------
                # Combined resolution score
                # --------------------------------------------

                combined = (
                    SEMANTIC_WEIGHT
                    * semantic
                    +
                    CONTEXT_WEIGHT
                    * ctx
                )


                # --------------------------------------------
                # Retain candidate
                # --------------------------------------------

                group_rows.append(
                    {
                        "instance_id":
                            instance_id,

                        "entity_type":
                            etype,

                        "mention_a":
                            a,

                        "mention_b":
                            b,

                        "name_a":
                            names[i],

                        "name_b":
                            names[j],

                        "semantic_similarity":
                            semantic,

                        "context_similarity":
                            ctx,

                        "combined_score":
                            combined,
                    }
                )


        # ====================================================
        # RELEASE BLOCK MEMORY IMMEDIATELY
        # ====================================================

        del sim_block

        gc.collect()


    # ========================================================
    # 7. CREATE GROUP DATAFRAME
    # ========================================================

    if group_rows:

        group_df = pd.DataFrame(
            group_rows,
            columns=CANDIDATE_COLUMNS
        )

    else:

        group_df = (
            empty_candidate_df()
        )


    # ========================================================
    # 8. CHECKPOINT GROUP IMMEDIATELY
    # ========================================================

    atomic_parquet_save(
        group_df,
        checkpoint_path
    )


    print(
        f"  ✓ Saved "
        f"{len(group_df):,} candidates"
    )

    print(
        f"  ✓ {checkpoint_path}"
    )


    # ========================================================
    # 9. FREE GROUP MEMORY
    # ========================================================

    del group_rows
    del group_df
    del E
    del keys
    del names

    gc.collect()


# ============================================================
# ALL GROUPS FINISHED
# ============================================================

print()

print(
    "=" * 65
)

print(
    "ALL CANDIDATE GROUPS COMPLETED"
)

print(
    "=" * 65
)


# ============================================================
# VERIFY ALL CHECKPOINTS EXIST
# ============================================================

missing_checkpoints = []


for (
    instance_id,
    etype
), group in groups:

    checkpoint_path = (
        group_checkpoint_path(
            instance_id,
            etype
        )
    )


    if not os.path.exists(
        checkpoint_path
    ):

        missing_checkpoints.append(
            (
                instance_id,
                etype
            )
        )


if missing_checkpoints:

    raise RuntimeError(
        "Some candidate groups are missing "
        f"checkpoints: {missing_checkpoints[:10]}"
    )


print(
    "✓ All group checkpoints verified."
)


# ============================================================
# 10. LOAD NON-EMPTY CHECKPOINTS
# ============================================================

candidate_frames = []


for (
    instance_id,
    etype
), group in tqdm(
    groups,
    desc="Loading candidate checkpoints"
):

    checkpoint_path = (
        group_checkpoint_path(
            instance_id,
            etype
        )
    )


    df = pd.read_parquet(
        checkpoint_path
    )


    if not df.empty:

        candidate_frames.append(
            df
        )


# ============================================================
# 11. COMBINE CANDIDATES
# ============================================================

if candidate_frames:

    candidates = pd.concat(
        candidate_frames,
        ignore_index=True
    )

else:

    candidates = (
        empty_candidate_df()
    )


del candidate_frames

gc.collect()


# ============================================================
# 12. SAVE FINAL COMBINED DATASET
# ============================================================

print()

print(
    "Saving final combined "
    "candidate dataset..."
)


atomic_parquet_save(
    candidates,
    CANDIDATES_PATH
)


print(
    "✓ Saved:",
    CANDIDATES_PATH
)


# ============================================================
# 13. SUMMARY
# ============================================================

print()

print(
    "=" * 65
)

print(
    "STEP 12 COMPLETE"
)

print(
    "=" * 65
)


print(
    "Semantic candidates retained:",
    f"{len(candidates):,}"
)


print(
    "Group checkpoints:",
    CANDIDATE_CACHE_DIR
)


print(
    "Combined candidate cache:",
    CANDIDATES_PATH
)


print()

print(
    "Semantic resolution excluded types:",
    sorted(
        SEMANTIC_EXCLUDED_TYPES
    )
)


# ============================================================
# 14. DISPLAY TOP CANDIDATES
# ============================================================

if not candidates.empty:

    display(
        candidates
        .sort_values(
            "combined_score",
            ascending=False
        )
        .head(30)
    )

Total instance/entity-type groups: 292
Checkpoint directory: /content/drive/MyDrive/entity_resolution_cache/candidate_group_cache

Largest entity groups:


,instance_id,entity_type,mentions,semantic_matching
165,MC-I01,ISSUE,10962,False
215,QTBUG-I01,ISSUE,2491,False
145,JRASERVER-I01,ISSUE,2134,False
30,CONFSERVER-I01,ISSUE,1852,False
135,JRACLOUD-I01,ISSUE,1548,False
155,JSDCLOUD-I01,ISSUE,1316,False
275,SPR-I01,ISSUE,1298,False
125,JBIDE-I01,ISSUE,1284,False
39,DAOS-I01,ISSUE,1073,False
245,SAK-I01,ISSUE,1053,False



Groups already cached: 165
Groups remaining: 127



Candidate groups:   0%|          | 0/292 [00:00<?, ?it/s]


Skipping semantic resolution: MC-I01 / ISSUE (10,962 mentions)
  ✓ Deterministic-ID type; semantic checkpoint saved.

Skipping semantic resolution: MC-I01 / MEETING (2 mentions)
  ✓ Deterministic-ID type; semantic checkpoint saved.

Processing: MC-I01 / MILESTONE (10 mentions)
  ✓ Saved 14 candidates
  ✓ /content/drive/MyDrive/entity_resolution_cache/candidate_group_cache/MC-I01__MILESTONE.parquet

Processing: MC-I01 / PERSON (288 mentions)
  ✓ Saved 37,202 candidates
  ✓ /content/drive/MyDrive/entity_resolution_cache/candidate_group_cache/MC-I01__PERSON.parquet

Skipping semantic resolution: MC-I01 / PROJECT (186 mentions)
  ✓ Deterministic-ID type; semantic checkpoint saved.

Processing: MC-I01 / REQUIREMENT (7 mentions)
  ✓ Saved 21 candidates
  ✓ /content/drive/MyDrive/entity_resolution_cache/candidate_group_cache/MC-I01__REQUIREMENT.parquet

Processing: MC-I01 / SYSTEM (15 mentions)
  ✓ Saved 88 candidates
  ✓ /content/drive/MyDrive/entity_resolution_cache/candidate_group_cache/M

Loading candidate checkpoints:   0%|          | 0/292 [00:00<?, ?it/s]


Saving final combined candidate dataset...
✓ Saved: /content/drive/MyDrive/entity_resolution_cache/semantic_candidates.parquet

STEP 12 COMPLETE
Semantic candidates retained: 8,620,612
Group checkpoints: /content/drive/MyDrive/entity_resolution_cache/candidate_group_cache
Combined candidate cache: /content/drive/MyDrive/entity_resolution_cache/semantic_candidates.parquet

Semantic resolution excluded types: ['ISSUE', 'MEETING', 'PROJECT']


,instance_id,entity_type,mention_a,mention_b,name_a,name_b,semantic_similarity,context_similarity,combined_score
1486239,CONFSERVER-I01,PERSON,CONFSERVER-I01||PRD||CONFSERVER-I01-PRD-V1||PR...,CONFSERVER-I01||PRD||CONFSERVER-I01-PRD-V1||PR...,customers,Customers,1.000000,1.0,1.000000
5129198,JRACLOUD-I01,PERSON,JRACLOUD-I01||PRD||JRACLOUD-I01-PRD-V1||PRD_JR...,JRACLOUD-I01||PRD||JRACLOUD-I01-PRD-V1||PRD_JR...,customers,Customers,1.000000,1.0,1.000000
8402683,SERVER-I01,MILESTONE,SERVER-I01||PRD||SERVER-I01-PRD-V1||PRD_SERVER...,SERVER-I01||PRD||SERVER-I01-PRD-V1||PRD_SERVER...,Storage sprint 2019-01-14,Storage sprint 2019-01-28,0.999672,1.0,0.999738
46943,AEROGEAR-I01,MILESTONE,AEROGEAR-I01||PRD||AEROGEAR-I01-PRD-V1||PRD_AE...,AEROGEAR-I01||PRD||AEROGEAR-I01-PRD-V1||PRD_AE...,ups-1.3.0,ups-1.2.0,0.999021,1.0,0.999217
2268337,ENTESB-I01,MILESTONE,ENTESB-I01||PRD||ENTESB-I01-PRD-V1||PRD_ENTESB...,ENTESB-I01||PRD||ENTESB-I01-PRD-V1||PRD_ENTESB...,OCP 3,OCP 4,0.997918,1.0,0.998334
7969092,OSSRH-I01,PERSON,OSSRH-I01||PRD||OSSRH-I01-PRD-V1||PRD_OSSRH-I0...,OSSRH-I01||PRD||OSSRH-I01-PRD-V1||PRD_OSSRH-I0...,requester,requesters,0.997855,1.0,0.998284
7631977,MC-I01,MILESTONE,MC-I01||PRD||MC-I01-PRD-V1||PRD_MC-I01-PRD-V1_...,MC-I01||PRD||MC-I01-PRD-V1||PRD_MC-I01-PRD-V1_...,20w12a,20w13a,0.997777,1.0,0.998222
1871182,DAOS-I01,MILESTONE,DAOS-I01||PRD||DAOS-I01-PRD-V1||PRD_DAOS-I01-P...,DAOS-I01||PRD||DAOS-I01-PRD-V1||PRD_DAOS-I01-P...,1.0,1.2,0.997767,1.0,0.998214
1486381,CONFSERVER-I01,SYSTEM,CONFSERVER-I01||MEETING||CONFSERVER-I01-M001||...,CONFSERVER-I01||MEETING||CONFSERVER-I01-M001||...,5.7.1,5.9.2,0.997368,1.0,0.997894
2644624,HBASE-I01,FEATURE,HBASE-I01||MEETING||HBASE-I01-M001||MEETING_HB...,HBASE-I01||MEETING||HBASE-I01-M001||MEETING_HB...,Phase 1,Phase 2,0.997190,1.0,0.997752


In [12]:
# ============================================================
# STEP 12B: CANDIDATE VALIDATION + SAFE PRUNING
#
# Purpose:
#   Filter MiniLM candidate pairs BEFORE clustering.
#
# This stage:
#   1. Removes entity types unsafe for semantic resolution
#   2. Rejects conflicting version/date/number identities
#   3. Rejects obvious generic pseudo-person entities
#   4. Adds lexical similarity signals
#   5. Separates SAFE AUTO-MERGE from AMBIGUOUS candidates
#   6. Saves the filtered result to Google Drive
#   7. Replaces `candidates` with the validated candidate set
#      so the existing Step 13 can continue unchanged.
#
# IMPORTANT:
#   This does NOT recompute MiniLM embeddings.
# ============================================================

import os
import gc
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

# Types where semantic similarity is too dangerous / unnecessary.
#
# ISSUE, PROJECT, MEETING were already excluded in Step 12.
# MILESTONE is added here because actual results showed pairs
# such as:
#
#   Phase 1 <-> Phase 2
#   EAP 6   <-> EAP 7
#   release 1.2 <-> release 1.3
#
# receiving extremely high semantic scores.
#
SEMANTIC_PRUNE_TYPES = {
    "ISSUE",
    "PROJECT",
    "MEETING",
    "MILESTONE",
}


# ------------------------------------------------------------
# Generic PERSON labels that should NOT be treated as named
# people during semantic entity resolution.
#
# We do NOT delete the underlying entity from the dataset.
# We only prevent semantic auto-merging based on these names.
# ------------------------------------------------------------

GENERIC_PERSON_TERMS = {
    "customer",
    "customers",
    "user",
    "users",
    "requester",
    "requesters",
    "developer",
    "developers",
    "dev",
    "admin",
    "administrator",
    "administrators",
    "manager",
    "managers",
    "engineer",
    "engineers",
    "owner",
    "owners",
    "client",
    "clients",
    "stakeholder",
    "stakeholders",
    "team",
    "teams",
    "staff",
    "member",
    "members",
    "participant",
    "participants",
    "author",
    "authors",
    "reporter",
    "reporters",
    "assignee",
    "assignees",
}


# ------------------------------------------------------------
# Thresholds
#
# These are deliberately conservative.
# ------------------------------------------------------------

# Semantic score required for non-exact auto-merge.
SAFE_SEMANTIC_THRESHOLD = 0.92

# Combined semantic/context score required.
SAFE_COMBINED_THRESHOLD = 0.93

# Lexical similarity required for semantic auto-merge.
SAFE_LEXICAL_THRESHOLD = 0.72

# More conservative PERSON thresholds.
PERSON_SEMANTIC_THRESHOLD = 0.95
PERSON_COMBINED_THRESHOLD = 0.95
PERSON_LEXICAL_THRESHOLD = 0.80


# ------------------------------------------------------------
# Candidate review threshold
#
# Pairs below this combined score are not useful enough to
# retain after structural validation.
# ------------------------------------------------------------

VALIDATED_REVIEW_THRESHOLD = 0.80


# ============================================================
# CACHE PATHS
# ============================================================

VALIDATED_CANDIDATES_PATH = os.path.join(
    ER_CACHE_DIR,
    "validated_semantic_candidates.parquet"
)

AUTO_MERGE_PATH = os.path.join(
    ER_CACHE_DIR,
    "safe_auto_merge_candidates.parquet"
)

AMBIGUOUS_PATH = os.path.join(
    ER_CACHE_DIR,
    "ambiguous_validated_candidates.parquet"
)

REJECTED_SUMMARY_PATH = os.path.join(
    ER_CACHE_DIR,
    "candidate_rejection_summary.csv"
)


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

def er_normalize_text(value):
    """
    Lightweight normalization used only for candidate
    validation.

    Does not modify original entity names.
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    # Standardize separators.
    value = re.sub(
        r"[_\-/]+",
        " ",
        value
    )

    # Remove most punctuation.
    value = re.sub(
        r"[^\w\s.]",
        " ",
        value
    )

    # Collapse whitespace.
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ============================================================
# TOKENIZATION
# ============================================================

def er_tokens(value):

    normalized = er_normalize_text(
        value
    )

    return {
        token
        for token in normalized.split()
        if token
    }


# ============================================================
# LEXICAL SIMILARITY
# ============================================================

def lexical_similarity(a, b):
    """
    Combination of:
        - character sequence similarity
        - token Jaccard similarity

    Returns 0..1.
    """

    a_norm = er_normalize_text(a)
    b_norm = er_normalize_text(b)

    if not a_norm or not b_norm:
        return 0.0

    if a_norm == b_norm:
        return 1.0


    # Character similarity.
    char_score = SequenceMatcher(
        None,
        a_norm,
        b_norm
    ).ratio()


    # Token similarity.
    a_tokens = set(
        a_norm.split()
    )

    b_tokens = set(
        b_norm.split()
    )


    union = (
        a_tokens | b_tokens
    )

    intersection = (
        a_tokens & b_tokens
    )


    if union:

        token_score = (
            len(intersection)
            / len(union)
        )

    else:

        token_score = 0.0


    # Slightly favor token overlap.
    return (
        0.45 * char_score
        +
        0.55 * token_score
    )


# ============================================================
# EXTRACT NUMERIC/VERSION TOKENS
# ============================================================

def numeric_identity_tokens(value):
    """
    Extract number-like identity markers.

    Examples:

        "EAP 6"
            -> {"6"}

        "Phase 2"
            -> {"2"}

        "ups-1.3.0"
            -> {"1.3.0"}

        "2019-01-14"
            -> {"2019", "01", "14"}

    These are used conservatively to detect obvious identity
    conflicts.
    """

    text = er_normalize_text(
        value
    )

    return set(
        re.findall(
            r"\b\d+(?:\.\d+)*\b",
            text
        )
    )


# ============================================================
# DATE EXTRACTION
# ============================================================

def date_identity_tokens(value):
    """
    Detect common date forms.

    Examples:
        2019-01-14
        2019/01/14
        2019.01.14
    """

    if pd.isna(value):
        return set()

    text = str(value).lower()

    dates = re.findall(
        r"\b\d{4}[-/.]\d{1,2}[-/.]\d{1,2}\b",
        text
    )

    return set(dates)


# ============================================================
# VERSION EXTRACTION
# ============================================================

def version_identity_tokens(value):
    """
    Detect version-like patterns.

    Examples:
        1.2
        1.3.0
        v2.1
        version 6
        release 7
    """

    if pd.isna(value):
        return set()

    text = str(value).lower()

    versions = set()


    # v1.2 / 1.2.3
    for match in re.findall(
        r"\bv?\d+\.\d+(?:\.\d+)*\b",
        text
    ):

        versions.add(
            match.lstrip("v")
        )


    # explicit "version 6", "release 7"
    for match in re.findall(
        r"\b(?:version|release)\s+([0-9]+(?:\.[0-9]+)*)",
        text
    ):

        versions.add(
            match
        )


    return versions


# ============================================================
# PHASE / MILESTONE-LIKE NUMBER EXTRACTION
# ============================================================

def labelled_number_tokens(value):
    """
    Detect labels whose numbers usually distinguish identity.

    Examples:
        Phase 1
        Phase 2

        Sprint 14
        Sprint 15

        M1
        M2

        ER1
        ER2
    """

    if pd.isna(value):
        return set()

    text = str(value).lower()

    tokens = set()


    # phase 1, sprint 2, iteration 3
    patterns = [
        r"\b(phase)\s*[-:]?\s*(\d+)\b",
        r"\b(sprint)\s*[-:]?\s*(\d+)\b",
        r"\b(iteration)\s*[-:]?\s*(\d+)\b",
        r"\b(milestone)\s*[-:]?\s*(\d+)\b",
    ]


    for pattern in patterns:

        for label, number in re.findall(
            pattern,
            text
        ):

            tokens.add(
                f"{label}:{number}"
            )


    # Short milestone/release labels such as:
    # M1, M2, ER1, ER2
    for prefix, number in re.findall(
        r"\b([a-z]{1,4})(\d+)\b",
        text
    ):

        tokens.add(
            f"{prefix}:{number}"
        )


    return tokens


# ============================================================
# STRUCTURAL IDENTITY CONFLICT
# ============================================================

def has_identity_conflict(name_a, name_b):
    """
    Return True when two highly similar strings contain
    clearly conflicting identity-bearing values.

    Examples rejected:

        EAP 6       vs EAP 7
        Phase 1     vs Phase 2
        M1          vs M2
        ups-1.2.0   vs ups-1.3.0
        sprint 2019-01-14 vs sprint 2019-01-28

    Important:
    Different numbers are NOT automatically considered a
    conflict unless the surrounding lexical structure is
    sufficiently similar.
    """

    a_norm = er_normalize_text(
        name_a
    )

    b_norm = er_normalize_text(
        name_b
    )


    if not a_norm or not b_norm:
        return False


    if a_norm == b_norm:
        return False


    # --------------------------------------------------------
    # DATE CONFLICT
    # --------------------------------------------------------

    dates_a = date_identity_tokens(
        name_a
    )

    dates_b = date_identity_tokens(
        name_b
    )


    if (
        dates_a
        and dates_b
        and dates_a != dates_b
    ):

        return True


    # --------------------------------------------------------
    # VERSION CONFLICT
    # --------------------------------------------------------

    versions_a = version_identity_tokens(
        name_a
    )

    versions_b = version_identity_tokens(
        name_b
    )


    if (
        versions_a
        and versions_b
        and versions_a != versions_b
    ):

        return True


    # --------------------------------------------------------
    # LABELLED NUMBER CONFLICT
    # --------------------------------------------------------

    labels_a = labelled_number_tokens(
        name_a
    )

    labels_b = labelled_number_tokens(
        name_b
    )


    if (
        labels_a
        and labels_b
        and labels_a != labels_b
    ):

        return True


    # --------------------------------------------------------
    # GENERAL NUMBER CONFLICT
    #
    # Only apply when the non-numeric lexical structure is
    # highly similar.
    # --------------------------------------------------------

    nums_a = numeric_identity_tokens(
        name_a
    )

    nums_b = numeric_identity_tokens(
        name_b
    )


    if (
        nums_a
        and nums_b
        and nums_a != nums_b
    ):

        # Remove numbers and compare remaining text.
        a_without_nums = re.sub(
            r"\b\d+(?:\.\d+)*\b",
            "",
            a_norm
        )

        b_without_nums = re.sub(
            r"\b\d+(?:\.\d+)*\b",
            "",
            b_norm
        )


        a_without_nums = re.sub(
            r"\s+",
            " ",
            a_without_nums
        ).strip()

        b_without_nums = re.sub(
            r"\s+",
            " ",
            b_without_nums
        ).strip()


        if (
            a_without_nums
            and
            a_without_nums
            == b_without_nums
        ):

            return True


    return False


# ============================================================
# GENERIC PERSON DETECTION
# ============================================================

def is_generic_person_name(value):

    normalized = er_normalize_text(
        value
    )


    if not normalized:
        return True


    # Exact generic term.
    if normalized in GENERIC_PERSON_TERMS:
        return True


    tokens = set(
        normalized.split()
    )


    # One-word generic role.
    if (
        len(tokens) == 1
        and
        next(iter(tokens))
        in GENERIC_PERSON_TERMS
    ):

        return True


    return False


# ============================================================
# ABBREVIATION / ALIAS SIGNAL
# ============================================================

def acronym(value):

    tokens = [
        token
        for token in er_normalize_text(
            value
        ).split()
        if token
    ]


    if len(tokens) < 2:
        return ""


    return "".join(
        token[0]
        for token in tokens
        if token
    )


def possible_acronym_match(a, b):
    """
    Examples:
        Authentication Service <-> AS
        Quality Assurance Team <-> QA
    """

    a_norm = (
        er_normalize_text(a)
        .replace(" ", "")
    )

    b_norm = (
        er_normalize_text(b)
        .replace(" ", "")
    )


    a_acronym = acronym(a)
    b_acronym = acronym(b)


    if (
        a_acronym
        and
        a_acronym == b_norm
    ):

        return True


    if (
        b_acronym
        and
        b_acronym == a_norm
    ):

        return True


    return False


# ============================================================
# CANDIDATE CLASSIFICATION
# ============================================================

def validate_candidate(row):
    """
    Returns:

        decision
        reason
        lexical_similarity

    Decisions:
        AUTO_MERGE
        AMBIGUOUS
        REJECT
    """

    etype = str(
        row["entity_type"]
    ).upper()


    name_a = str(
        row["name_a"]
    )

    name_b = str(
        row["name_b"]
    )


    semantic = float(
        row["semantic_similarity"]
    )

    combined = float(
        row["combined_score"]
    )


    norm_a = er_normalize_text(
        name_a
    )

    norm_b = er_normalize_text(
        name_b
    )


    # ========================================================
    # RULE 1: TYPES UNSAFE FOR SEMANTIC MERGING
    # ========================================================

    if etype in SEMANTIC_PRUNE_TYPES:

        return (
            "REJECT",
            "semantic_resolution_not_applicable",
            0.0
        )


    # ========================================================
    # RULE 2: EMPTY NAMES
    # ========================================================

    if not norm_a or not norm_b:

        return (
            "REJECT",
            "empty_name",
            0.0
        )


    # ========================================================
    # RULE 3: EXACT NORMALIZED NAME
    #
    # Usually already handled by Step 7, but if any survive,
    # this is very strong identity evidence.
    # ========================================================

    if norm_a == norm_b:

        return (
            "AUTO_MERGE",
            "exact_normalized_name",
            1.0
        )


    # ========================================================
    # RULE 4: CONFLICTING VERSION/DATE/NUMBER
    # ========================================================

    if has_identity_conflict(
        name_a,
        name_b
    ):

        return (
            "REJECT",
            "conflicting_identity_marker",
            0.0
        )


    # ========================================================
    # RULE 5: GENERIC PERSON LABELS
    # ========================================================

    if etype == "PERSON":

        if (
            is_generic_person_name(name_a)
            or
            is_generic_person_name(name_b)
        ):

            return (
                "REJECT",
                "generic_person_label",
                0.0
            )


    # ========================================================
    # CALCULATE LEXICAL SIGNAL
    # ========================================================

    lexical = lexical_similarity(
        name_a,
        name_b
    )


    # ========================================================
    # RULE 6: ACRONYM / ALIAS
    # ========================================================

    acronym_match = (
        possible_acronym_match(
            name_a,
            name_b
        )
    )


    # ========================================================
    # RULE 7: PERSON - STRICTER AUTO MERGE
    # ========================================================

    if etype == "PERSON":

        if (
            semantic
            >= PERSON_SEMANTIC_THRESHOLD
            and
            combined
            >= PERSON_COMBINED_THRESHOLD
            and
            lexical
            >= PERSON_LEXICAL_THRESHOLD
        ):

            return (
                "AUTO_MERGE",
                "high_confidence_person_match",
                lexical
            )


        # Keep plausible person aliases for later review,
        # but DO NOT automatically merge them.
        if (
            combined
            >= VALIDATED_REVIEW_THRESHOLD
            and
            lexical >= 0.55
        ):

            return (
                "AMBIGUOUS",
                "possible_person_alias",
                lexical
            )


        return (
            "REJECT",
            "insufficient_person_evidence",
            lexical
        )


    # ========================================================
    # RULE 8: ACRONYM MATCH
    # ========================================================

    if (
        acronym_match
        and
        semantic >= 0.85
    ):

        return (
            "AUTO_MERGE",
            "acronym_alias_match",
            lexical
        )


    # ========================================================
    # RULE 9: GENERAL HIGH-CONFIDENCE MATCH
    # ========================================================

    if (
        semantic
        >= SAFE_SEMANTIC_THRESHOLD
        and
        combined
        >= SAFE_COMBINED_THRESHOLD
        and
        lexical
        >= SAFE_LEXICAL_THRESHOLD
    ):

        return (
            "AUTO_MERGE",
            "high_semantic_lexical_match",
            lexical
        )


    # ========================================================
    # RULE 10: AMBIGUOUS BUT PLAUSIBLE
    # ========================================================

    if (
        combined
        >= VALIDATED_REVIEW_THRESHOLD
        and
        lexical >= 0.50
    ):

        return (
            "AMBIGUOUS",
            "plausible_but_not_safe",
            lexical
        )


    # ========================================================
    # OTHERWISE REJECT
    # ========================================================

    return (
        "REJECT",
        "insufficient_identity_evidence",
        lexical
    )


# ============================================================
# LOAD CANDIDATES IF NECESSARY
# ============================================================

if "candidates" not in globals():

    print(
        "Candidates not currently in RAM."
    )

    print(
        "Loading Step 12 cache..."
    )

    candidates = pd.read_parquet(
        CANDIDATES_PATH
    )


print(
    "Input candidates:",
    f"{len(candidates):,}"
)


# ============================================================
# PROCESS IN CHUNKS
#
# 8.6M rows is large, so do NOT use candidates.apply(...)
# across the entire DataFrame at once.
# ============================================================

VALIDATION_CHUNK_SIZE = 100_000


validated_chunks = []


total_rows = len(
    candidates
)


for start in tqdm(
    range(
        0,
        total_rows,
        VALIDATION_CHUNK_SIZE
    ),
    desc="Validating candidates"
):

    end = min(
        start + VALIDATION_CHUNK_SIZE,
        total_rows
    )


    chunk = (
        candidates
        .iloc[start:end]
        .copy()
    )


    decisions = []
    reasons = []
    lexical_scores = []


    for row in chunk.to_dict(
        orient="records"
    ):

        (
            decision,
            reason,
            lexical
        ) = validate_candidate(
            row
        )


        decisions.append(
            decision
        )

        reasons.append(
            reason
        )

        lexical_scores.append(
            lexical
        )


    chunk[
        "validation_decision"
    ] = decisions


    chunk[
        "validation_reason"
    ] = reasons


    chunk[
        "lexical_similarity"
    ] = np.asarray(
        lexical_scores,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Keep only AUTO_MERGE + AMBIGUOUS rows in RAM.
    #
    # REJECT rows are summarized rather than retained.
    # --------------------------------------------------------

    keep_chunk = chunk[
        chunk[
            "validation_decision"
        ].isin(
            [
                "AUTO_MERGE",
                "AMBIGUOUS"
            ]
        )
    ].copy()


    if not keep_chunk.empty:

        validated_chunks.append(
            keep_chunk
        )


    del chunk
    del decisions
    del reasons
    del lexical_scores
    del keep_chunk

    gc.collect()


# ============================================================
# COMBINE VALIDATED CANDIDATES
# ============================================================

if validated_chunks:

    validated_candidates = pd.concat(
        validated_chunks,
        ignore_index=True
    )

else:

    validated_candidates = pd.DataFrame(
        columns=list(candidates.columns)
        + [
            "validation_decision",
            "validation_reason",
            "lexical_similarity"
        ]
    )


del validated_chunks

gc.collect()


# ============================================================
# SPLIT SAFE AUTO-MERGES AND AMBIGUOUS PAIRS
# ============================================================

safe_auto_merge_candidates = (
    validated_candidates[
        validated_candidates[
            "validation_decision"
        ]
        == "AUTO_MERGE"
    ]
    .copy()
)


ambiguous_validated_candidates = (
    validated_candidates[
        validated_candidates[
            "validation_decision"
        ]
        == "AMBIGUOUS"
    ]
    .copy()
)


# ============================================================
# SAVE RESULTS
# ============================================================

print()
print(
    "Saving Step 12B outputs..."
)


validated_candidates.to_parquet(
    VALIDATED_CANDIDATES_PATH,
    index=False
)


safe_auto_merge_candidates.to_parquet(
    AUTO_MERGE_PATH,
    index=False
)


ambiguous_validated_candidates.to_parquet(
    AMBIGUOUS_PATH,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

original_count = len(
    candidates
)

validated_count = len(
    validated_candidates
)

auto_count = len(
    safe_auto_merge_candidates
)

ambiguous_count = len(
    ambiguous_validated_candidates
)


removed_count = (
    original_count
    - validated_count
)


print()
print(
    "=" * 70
)

print(
    "STEP 12B COMPLETE"
)

print(
    "=" * 70
)


print(
    f"Original candidates:     "
    f"{original_count:,}"
)

print(
    f"Rejected/pruned:         "
    f"{removed_count:,}"
)

print(
    f"Candidates retained:     "
    f"{validated_count:,}"
)

print(
    f"Safe AUTO_MERGE:         "
    f"{auto_count:,}"
)

print(
    f"AMBIGUOUS:               "
    f"{ambiguous_count:,}"
)


if original_count > 0:

    reduction_pct = (
        100
        * removed_count
        / original_count
    )

    print(
        f"Candidate reduction:     "
        f"{reduction_pct:.2f}%"
    )


print()

print(
    "Validated cache:"
)

print(
    VALIDATED_CANDIDATES_PATH
)


print()

print(
    "Safe auto-merge cache:"
)

print(
    AUTO_MERGE_PATH
)


print()

print(
    "Ambiguous cache:"
)

print(
    AMBIGUOUS_PATH
)


# ============================================================
# DIAGNOSTIC: DECISIONS BY ENTITY TYPE
# ============================================================

print()
print(
    "Retained candidates by entity type / decision:"
)


if not validated_candidates.empty:

    decision_summary = (
        validated_candidates
        .groupby(
            [
                "entity_type",
                "validation_decision"
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
        .sort_values(
            "count",
            ascending=False
        )
    )


    display(
        decision_summary
    )


# ============================================================
# DIAGNOSTIC: TOP SAFE MERGES
# ============================================================

print()
print(
    "Top proposed SAFE AUTO-MERGES:"
)


if not safe_auto_merge_candidates.empty:

    display(
        safe_auto_merge_candidates[
            [
                "instance_id",
                "entity_type",
                "name_a",
                "name_b",
                "semantic_similarity",
                "context_similarity",
                "lexical_similarity",
                "combined_score",
                "validation_reason"
            ]
        ]
        .sort_values(
            "combined_score",
            ascending=False
        )
        .head(50)
    )


# ============================================================
# DIAGNOSTIC: TOP AMBIGUOUS PAIRS
# ============================================================

print()
print(
    "Top AMBIGUOUS pairs:"
)


if not ambiguous_validated_candidates.empty:

    display(
        ambiguous_validated_candidates[
            [
                "instance_id",
                "entity_type",
                "name_a",
                "name_b",
                "semantic_similarity",
                "context_similarity",
                "lexical_similarity",
                "combined_score",
                "validation_reason"
            ]
        ]
        .sort_values(
            "combined_score",
            ascending=False
        )
        .head(50)
    )


# ============================================================
# IMPORTANT:
# MAKE EXISTING STEP 13 COMPATIBLE
# ============================================================
#
# Your existing Step 13 expects a DataFrame named:
#
#     candidates
#
# We replace it with ONLY the safe auto-merge candidates.
#
# This means Step 13 cannot accidentally cluster ambiguous
# candidates.
#
# Ambiguous candidates remain available separately in:
#
#     ambiguous_validated_candidates
#
# ============================================================

print()
print(
    "Preparing safe candidate set for Step 13..."
)


# Free the original 8.6M candidate DataFrame before replacing.
del candidates

gc.collect()


candidates = (
    safe_auto_merge_candidates
    .reset_index(
        drop=True
    )
)


print(
    "Step 13 candidate count:",
    f"{len(candidates):,}"
)


print()
print(
    "✓ `candidates` now contains SAFE AUTO-MERGE "
    "pairs only."
)

print(
    "✓ `ambiguous_validated_candidates` remains "
    "separate and will NOT be automatically clustered."
)

Input candidates: 8,620,612


Validating candidates:   0%|          | 0/87 [00:00<?, ?it/s]


Saving Step 12B outputs...

STEP 12B COMPLETE
Original candidates:     8,620,612
Rejected/pruned:         8,611,811
Candidates retained:     8,801
Safe AUTO_MERGE:         1,514
AMBIGUOUS:               7,287
Candidate reduction:     99.90%

Validated cache:
/content/drive/MyDrive/entity_resolution_cache/validated_semantic_candidates.parquet

Safe auto-merge cache:
/content/drive/MyDrive/entity_resolution_cache/safe_auto_merge_candidates.parquet

Ambiguous cache:
/content/drive/MyDrive/entity_resolution_cache/ambiguous_validated_candidates.parquet

Retained candidates by entity type / decision:


,entity_type,validation_decision,count
0,COMPONENT,AMBIGUOUS,7064
1,COMPONENT,AUTO_MERGE,1390
5,PERSON,AUTO_MERGE,99
2,FEATURE,AMBIGUOUS,95
6,SYSTEM,AMBIGUOUS,81
8,TEAM,AMBIGUOUS,34
4,PERSON,AMBIGUOUS,13
3,FEATURE,AUTO_MERGE,10
7,SYSTEM,AUTO_MERGE,8
9,TEAM,AUTO_MERGE,7



Top proposed SAFE AUTO-MERGES:


,instance_id,entity_type,name_a,name_b,semantic_similarity,context_similarity,lexical_similarity,combined_score,validation_reason
792,CONFSERVER-I01,PERSON,customers,Customers,1.000000,1.000000,1.000000,1.000000,exact_normalized_name
4467,JRACLOUD-I01,PERSON,customers,Customers,1.000000,1.000000,1.000000,1.000000,exact_normalized_name
4731,JRASERVER-I01,FEATURE,priority scheme support,priority scheme,0.990395,1.000000,0.721930,0.992316,high_semantic_lexical_match
1160,DAOS-I01,COMPONENT,daos_server,daos_io_server,0.984714,1.000000,0.762667,0.987771,high_semantic_lexical_match
5194,MNG-I01,FEATURE,new sync contexts,sync contexts,0.989087,0.944444,0.756667,0.980158,high_semantic_lexical_match
713,AMBARI-I01,TEAM,Contributor team,Vendor contributor team,0.973317,1.000000,0.735897,0.978653,high_semantic_lexical_match
781,CONFSERVER-I01,FEATURE,upgrade-path coverage,upgrade coverage,0.982724,0.916667,0.755856,0.969512,high_semantic_lexical_match
4489,JRASERVER-I01,COMPONENT,Migration - Cloud to Server,Migration - Server to Cloud,0.990291,0.875000,0.838000,0.967233,high_semantic_lexical_match
4370,JRACLOUD-I01,COMPONENT,Backup & Restore - Import From JIRA Cloud,Backup & Restore - Import From JIRA Server,0.978587,0.875000,0.776857,0.957869,high_semantic_lexical_match
4252,JRACLOUD-I01,COMPONENT,Backup & Restore - Import From JIRA Cloud,Backup & Restore - Import From JIRA Server,0.977969,0.875000,0.776857,0.957375,high_semantic_lexical_match



Top AMBIGUOUS pairs:


,instance_id,entity_type,name_a,name_b,semantic_similarity,context_similarity,lexical_similarity,combined_score,validation_reason
4032,JBDS-I01,FEATURE,AERI,AERI feature,0.995849,1.000000,0.500000,0.996679,plausible_but_not_safe
2543,EVG-I01,TEAM,Evergreen,Evergreen team,0.995108,1.000000,0.627174,0.996086,plausible_but_not_safe
4055,JBEAP-I01,TEAM,dev,Core dev,0.993267,1.000000,0.520455,0.994614,plausible_but_not_safe
6307,SAK-I01,COMPONENT,Admin UI,admin,0.986334,1.000000,0.621154,0.989067,plausible_but_not_safe
5167,JSDCLOUD-I01,SYSTEM,Jira Service Management Cloud,Jira Service Desk Cloud,0.985882,1.000000,0.676154,0.988705,plausible_but_not_safe
2583,HBASE-I01,COMPONENT,replication state,replication table,0.984056,1.000000,0.580392,0.987245,plausible_but_not_safe
5172,JSDCLOUD-I01,SYSTEM,Jira,Jira-created,0.982863,1.000000,0.500000,0.986291,plausible_but_not_safe
2596,HBASE-I01,SYSTEM,hbase-spark,spark,0.977489,1.000000,0.556250,0.981991,plausible_but_not_safe
6116,QTBUG-I01,SYSTEM,MinGW,MinGW 7.3,0.991043,0.931818,0.596429,0.979198,plausible_but_not_safe
1185,DOCS-I01,SYSTEM,PRD,The PRD,0.990983,0.916667,0.545000,0.976119,plausible_but_not_safe



Preparing safe candidate set for Step 13...
Step 13 candidate count: 1,514

✓ `candidates` now contains SAFE AUTO-MERGE pairs only.
✓ `ambiguous_validated_candidates` remains separate and will NOT be automatically clustered.


In [13]:
# ============================================================
# STEP 12C: FINAL AUTO-MERGE SAFETY FILTER
# ============================================================
#
# Purpose:
#   Apply a final conservative filter before Step 13 clustering.
#
# We prefer false negatives over false-positive entity merges.
# Ambiguous/rejected pairs remain separate entities in the KG.
# ============================================================

import re
import gc
import pandas as pd


# ============================================================
# 1. GENERIC / INVALID PERSON LABELS
# ============================================================

GENERIC_OR_INVALID_PERSON_TERMS = {
    "customer",
    "customers",
    "user",
    "users",
    "requester",
    "requesters",
    "developer",
    "developers",
    "dev",
    "admin",
    "administrator",
    "manager",
    "managers",
    "engineer",
    "engineers",
    "owner",
    "owners",
    "client",
    "clients",
    "stakeholder",
    "stakeholders",
    "participant",
    "participants",
    "facilitator",
    "facilitators",
    "reporter",
    "reporters",
    "assignee",
    "assignees",
    "notification",
    "notifications",
    "windows",
    "platform",
}


def normalize_final(value):

    value = str(value).strip().lower()

    value = re.sub(
        r"[^a-z0-9\s\-']",
        "",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def invalid_person_label(value):

    value = normalize_final(value)

    if not value:
        return True

    if value in GENERIC_OR_INVALID_PERSON_TERMS:
        return True

    # Reject obvious dangling extraction fragments.
    original = str(value).strip()

    if len(original) <= 1:
        return True

    return False


# ============================================================
# 2. DIRECTIONAL / CONTRAST TERMS
# ============================================================

CONTRAST_GROUPS = [
    {"cloud", "server"},
    {"import", "export"},
    {"source", "target"},
    {"from", "to"},
    {"input", "output"},
    {"static", "dynamic"},
    {"enable", "disable"},
    {"enabled", "disabled"},
    {"start", "stop"},
    {"create", "delete"},
    {"add", "remove"},
]


def token_set(value):

    return set(
        normalize_final(value)
        .split()
    )


def has_contrast_conflict(a, b):

    ta = token_set(a)
    tb = token_set(b)

    for group in CONTRAST_GROUPS:

        a_terms = ta & group
        b_terms = tb & group

        # Each name contains a different member
        # of the same contrast group.
        if (
            a_terms
            and b_terms
            and a_terms != b_terms
        ):
            return True

    return False


# ============================================================
# 3. FINAL SAFETY DECISION
# ============================================================

def final_safe_merge(row):

    etype = str(
        row["entity_type"]
    ).upper()

    name_a = str(
        row["name_a"]
    )

    name_b = str(
        row["name_b"]
    )

    norm_a = normalize_final(name_a)
    norm_b = normalize_final(name_b)


    # --------------------------------------------------------
    # PERSON
    # --------------------------------------------------------

    if etype == "PERSON":

        if (
            invalid_person_label(name_a)
            or invalid_person_label(name_b)
        ):
            return False

        # Exact names are acceptable after generic-name filter.
        if norm_a == norm_b:
            return True

        # For this prototype, do NOT automatically merge
        # non-exact PERSON names.
        #
        # Example:
        #   John
        #   John Smith
        #
        # can remain ambiguous rather than risking identity
        # collapse.
        return False


    # --------------------------------------------------------
    # DIRECTIONAL / CONTRAST CONFLICT
    # --------------------------------------------------------

    if has_contrast_conflict(
        name_a,
        name_b
    ):
        return False


    # --------------------------------------------------------
    # EXACT NORMALIZED NAME
    # --------------------------------------------------------

    if norm_a == norm_b:
        return True


    # --------------------------------------------------------
    # NON-EXACT COMPONENT MATCHES
    #
    # Be deliberately conservative.
    #
    # Components often have closely related names that refer
    # to different modules/subsystems.
    # --------------------------------------------------------

    if etype == "COMPONENT":

        # Require extremely strong evidence.
        return (
            float(row["semantic_similarity"]) >= 0.97
            and
            float(row["lexical_similarity"]) >= 0.85
        )


    # --------------------------------------------------------
    # OTHER TYPES
    #
    # Keep Step 12B's already-conservative AUTO_MERGE result.
    # --------------------------------------------------------

    return True


# ============================================================
# APPLY FILTER
# ============================================================

before = len(candidates)


safe_mask = candidates.apply(
    final_safe_merge,
    axis=1
)


removed_by_final_filter = (
    candidates[
        ~safe_mask
    ]
    .copy()
)


candidates = (
    candidates[
        safe_mask
    ]
    .copy()
    .reset_index(drop=True)
)


after = len(candidates)


# ============================================================
# SAVE
# ============================================================

FINAL_SAFE_PATH = os.path.join(
    ER_CACHE_DIR,
    "final_safe_auto_merge_candidates.parquet"
)

FINAL_REJECTED_PATH = os.path.join(
    ER_CACHE_DIR,
    "final_safety_rejected_candidates.parquet"
)


candidates.to_parquet(
    FINAL_SAFE_PATH,
    index=False
)


removed_by_final_filter.to_parquet(
    FINAL_REJECTED_PATH,
    index=False
)


gc.collect()


# ============================================================
# SUMMARY
# ============================================================

print("=" * 65)
print("STEP 12C COMPLETE")
print("=" * 65)

print(
    f"Step 12B AUTO_MERGE pairs: {before:,}"
)

print(
    f"Removed by final safety filter: "
    f"{before - after:,}"
)

print(
    f"Final Step 13 merge pairs: {after:,}"
)

print()

print(
    "Saved final safe candidates:"
)

print(
    FINAL_SAFE_PATH
)


# ============================================================
# INSPECT REMOVED PAIRS
# ============================================================

print()
print("Examples removed by final safety filter:")

if not removed_by_final_filter.empty:

    display(
        removed_by_final_filter[
            [
                "instance_id",
                "entity_type",
                "name_a",
                "name_b",
                "semantic_similarity",
                "lexical_similarity",
                "combined_score",
            ]
        ]
        .sort_values(
            "combined_score",
            ascending=False
        )
        .head(30)
    )


# ============================================================
# INSPECT FINAL MERGES
# ============================================================

print()
print("Final proposed merges:")

if not candidates.empty:

    display(
        candidates[
            [
                "instance_id",
                "entity_type",
                "name_a",
                "name_b",
                "semantic_similarity",
                "lexical_similarity",
                "combined_score",
            ]
        ]
        .sort_values(
            "combined_score",
            ascending=False
        )
        .head(50)
    )

STEP 12C COMPLETE
Step 12B AUTO_MERGE pairs: 1,514
Removed by final safety filter: 20
Final Step 13 merge pairs: 1,494

Saved final safe candidates:
/content/drive/MyDrive/entity_resolution_cache/final_safe_auto_merge_candidates.parquet

Examples removed by final safety filter:


,instance_id,entity_type,name_a,name_b,semantic_similarity,lexical_similarity,combined_score
19,CONFSERVER-I01,PERSON,customers,Customers,1.000000,1.000000,1.000000
972,JRACLOUD-I01,PERSON,customers,Customers,1.000000,1.000000,1.000000
22,DAOS-I01,COMPONENT,daos_server,daos_io_server,0.984714,0.762667,0.987771
973,JRASERVER-I01,COMPONENT,Migration - Cloud to Server,Migration - Server to Cloud,0.990291,0.838000,0.967233
965,JRACLOUD-I01,COMPONENT,Backup & Restore - Import From JIRA Cloud,Backup & Restore - Import From JIRA Server,0.978587,0.776857,0.957869
964,JRACLOUD-I01,COMPONENT,Backup & Restore - Import From JIRA Cloud,Backup & Restore - Import From JIRA Server,0.977969,0.776857,0.957375
966,JRACLOUD-I01,COMPONENT,Backup & Restore - Import From JIRA Cloud,Backup & Restore - Import From JIRA Server,0.977433,0.776857,0.956946
968,JRACLOUD-I01,PERSON,Notifications),Notifications),0.998784,1.000000,0.949027
1,AMBARI-I01,PERSON,Dev,Dev,0.998607,1.000000,0.948885
1478,QTCREATORBUG-I01,PERSON,Windows),Windows),0.998552,1.000000,0.948842



Final proposed merges:


,instance_id,entity_type,name_a,name_b,semantic_similarity,lexical_similarity,combined_score
958,JRASERVER-I01,FEATURE,priority scheme support,priority scheme,0.990395,0.721930,0.992316
972,MNG-I01,FEATURE,new sync contexts,sync contexts,0.989087,0.756667,0.980158
6,AMBARI-I01,TEAM,Contributor team,Vendor contributor team,0.973317,0.735897,0.978653
7,CONFSERVER-I01,FEATURE,upgrade-path coverage,upgrade coverage,0.982724,0.755856,0.969512
1466,SAK-I01,TEAM,QA,QA,0.999431,1.000000,0.949545
1461,QTCREATORBUG-I01,PERSON,Sam,Sam,0.999087,1.000000,0.949269
1484,SPARK-I01,PERSON,Alicia,Alicia,0.999044,1.000000,0.949235
1480,SPARK-I01,PERSON,Marcus,Marcus,0.998998,1.000000,0.949198
1481,SPARK-I01,PERSON,Priya,Priya,0.998947,1.000000,0.949157
29,ENTESB-I01,PERSON,Mara,Mara,0.998901,1.000000,0.949120


In [14]:
# ============================================================
# STEP 13: CONSERVATIVE SEMANTIC MERGING + AMBIGUOUS REVIEW EXPORT
# ============================================================
ambiguous_rows = []
if not candidates.empty:
    for row in candidates.sort_values('combined_score', ascending=False).to_dict('records'):
        a, b = row['mention_a'], row['mention_b']
        if uf.find(a) == uf.find(b):
            continue
        semantic = row['semantic_similarity']
        combined = row['combined_score']
        # Require BOTH strong semantic similarity and strong combined evidence.
        if semantic >= SEMANTIC_ACCEPT_THRESHOLD and combined >= COMBINED_ACCEPT_THRESHOLD:
            accept_pair(a, b, 'MINILM_CONTEXT', combined, semantic, row['context_similarity'])
        else:
            row['decision'] = 'REVIEW_NOT_MERGED'
            ambiguous_rows.append(row)

ambiguous = pd.DataFrame(ambiguous_rows)
ambiguous_path = OUTPUT_ROOT / 'ambiguous_pairs_for_review.csv'
ambiguous.to_csv(ambiguous_path, index=False)
print('Semantic merge edges:', sum(e['method']=='MINILM_CONTEXT' for e in resolution_edges))
print('Ambiguous pairs left unmerged:', len(ambiguous))
print('Saved:', ambiguous_path)

Semantic merge edges: 185
Ambiguous pairs left unmerged: 48
Saved: /content/drive/MyDrive/GC1/entity_resolution_output/ambiguous_pairs_for_review.csv


In [15]:
# ============================================================
# STEP 13B: CLUSTER SANITY CHECK
# ============================================================

from collections import defaultdict
import pandas as pd

# Build cluster membership
clusters = defaultdict(list)

for key in mentions["mention_key"]:
    root = uf.find(key)
    clusters[root].append(key)


# Lookup metadata
mention_lookup = (
    mentions
    .set_index("mention_key")
    .to_dict("index")
)


cluster_rows = []

for root, members in clusters.items():

    member_records = [
        mention_lookup[m]
        for m in members
    ]

    names = sorted({
        str(r["name"])
        for r in member_records
    })

    types = sorted({
        str(r["entity_type"])
        for r in member_records
    })

    instances = sorted({
        str(r["instance_id"])
        for r in member_records
    })

    cluster_rows.append({
        "cluster_root": root,
        "cluster_size": len(members),
        "entity_types": types,
        "instances": instances,
        "unique_names": len(names),
        "names": names[:20],
    })


cluster_diagnostics = pd.DataFrame(cluster_rows)


print("=" * 70)
print("ENTITY RESOLUTION CLUSTER DIAGNOSTICS")
print("=" * 70)

print(
    "Total mentions:",
    f"{len(mentions):,}"
)

print(
    "Canonical clusters:",
    f"{len(cluster_diagnostics):,}"
)

print(
    "Mentions merged away:",
    f"{len(mentions) - len(cluster_diagnostics):,}"
)

print()

print("Cluster size distribution:")

display(
    cluster_diagnostics[
        "cluster_size"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("cluster_size")
    .reset_index(name="number_of_clusters")
)


# ------------------------------------------------------------
# Largest clusters
# ------------------------------------------------------------

print()
print("Largest clusters:")

display(
    cluster_diagnostics
    .sort_values(
        "cluster_size",
        ascending=False
    )
    .head(30)
)


# ------------------------------------------------------------
# Check for cross-instance contamination
# ------------------------------------------------------------

cross_instance = cluster_diagnostics[
    cluster_diagnostics[
        "instances"
    ].apply(len) > 1
]

print()
print(
    "Cross-instance clusters:",
    len(cross_instance)
)

if not cross_instance.empty:

    display(
        cross_instance.head(30)
    )


# ------------------------------------------------------------
# Check for cross-type contamination
# ------------------------------------------------------------

cross_type = cluster_diagnostics[
    cluster_diagnostics[
        "entity_types"
    ].apply(len) > 1
]

print(
    "Cross-type clusters:",
    len(cross_type)
)

if not cross_type.empty:

    display(
        cross_type.head(30)
    )

ENTITY RESOLUTION CLUSTER DIAGNOSTICS
Total mentions: 61,392
Canonical clusters: 40,325
Mentions merged away: 21,067

Cluster size distribution:


,cluster_size,number_of_clusters
0,1,34972
1,2,3178
2,3,974
3,4,395
4,5,205
...,...,...
87,285,1
88,286,1
89,290,1
90,301,1



Largest clusters:


,cluster_root,cluster_size,entity_types,instances,unique_names,names
6836,SPR-I01||JIRA||SPR-17637||JIRA_SPR-17637_E001,303,[PROJECT],[SPR-I01],1,[SPR]
37494,RUBY-I01||JIRA||RUBY-2848||JIRA_RUBY-2848_E001,301,[PROJECT],[RUBY-I01],1,[RUBY]
21365,MONGOID-I01||JIRA||MONGOID-5080||JIRA_MONGOID-...,290,[PROJECT],[MONGOID-I01],1,[MONGOID]
24707,JRASERVER-I01||JIRA||JRASERVER-67869||JIRA_JRA...,286,[PROJECT],[JRASERVER-I01],1,[JRASERVER]
30833,JRACLOUD-I01||JIRA||JRACLOUD-77518||JIRA_JRACL...,285,[PROJECT],[JRACLOUD-I01],1,[JRACLOUD]
27637,CONFSERVER-I01||JIRA||CONFSERVER-51780||JIRA_C...,264,[PROJECT],[CONFSERVER-I01],1,[CONFSERVER]
35207,JSDCLOUD-I01||JIRA||JSDCLOUD-9774||JIRA_JSDCLO...,262,[PROJECT],[JSDCLOUD-I01],1,[JSDCLOUD]
5839,ENTESB-I01||JIRA||ENTESB-15364||JIRA_ENTESB-15...,260,[PROJECT],[ENTESB-I01],1,[ENTESB]
22130,QTBUG-I01||JIRA||QTBUG-74860||JIRA_QTBUG-74860...,253,[PROJECT],[QTBUG-I01],1,[QTBUG]
34377,NEXUS-I01||JIRA||NEXUS-21383||JIRA_NEXUS-21383...,252,[PROJECT],[NEXUS-I01],1,[NEXUS]



Cross-instance clusters: 0
Cross-type clusters: 0


In [20]:
# ============================================================
# STEP 14: ASSIGN STABLE COLLISION-SAFE CANONICAL IDs
# ============================================================
#
# IMPORTANT:
# Canonical IDs are based on the resolved UF cluster identity,
# NOT merely the canonical entity name.
#
# This guarantees:
#
#   1 UF cluster = 1 canonical_id
#
# even when separate unresolved entities share the same name.
# ============================================================

import hashlib
import re
import gc
from collections import defaultdict

import pandas as pd


# ============================================================
# 1. BUILD CURRENT UNION-FIND CLUSTERS
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/GC1/"
    "entity_resolution_output"
)

clusters = defaultdict(list)

for mention_key in mentions["mention_key"]:

    root = uf.find(
        mention_key
    )

    clusters[root].append(
        mention_key
    )


print(
    "Current Union-Find clusters:",
    len(clusters)
)


# ============================================================
# 2. MENTION LOOKUP
# ============================================================

mention_lookup = (
    mentions
    .set_index("mention_key")
    .to_dict("index")
)


# ============================================================
# 3. SLUG HELPER
# ============================================================

def canonical_slug(value, max_length=60):

    value = str(value).strip().lower()

    # Normalize separators/punctuation.
    value = re.sub(
        r"[^a-z0-9]+",
        "-",
        value
    )

    value = re.sub(
        r"-+",
        "-",
        value
    )

    value = value.strip("-")

    if not value:
        value = "unnamed"

    return value[:max_length]


# ============================================================
# 4. CHOOSE CANONICAL NAME
# ============================================================

def choose_canonical_name(member_records):
    """
    Choose a stable human-readable canonical name.

    Preference:
      1. Most frequently occurring normalized/original name
      2. Longer form when frequency ties
      3. Lexicographic tie-break for reproducibility
    """

    names = [
        str(record["name"]).strip()
        for record in member_records
        if str(record["name"]).strip()
    ]

    if not names:
        return "Unnamed"


    name_counts = pd.Series(
        names
    ).value_counts()


    max_count = (
        name_counts.max()
    )


    top_names = [
        name
        for name, count
        in name_counts.items()
        if count == max_count
    ]


    # Prefer informative/longer form.
    top_names = sorted(
        top_names,
        key=lambda x: (
            -len(x),
            x.lower(),
            x
        )
    )


    return top_names[0]


# ============================================================
# 5. CLUSTER FINGERPRINT
# ============================================================

def cluster_fingerprint(member_keys):
    """
    Generate a deterministic fingerprint from the complete
    membership of the resolved cluster.

    Sorting makes this independent of row order.
    """

    stable_cluster_string = "\n".join(
        sorted(
            map(str, member_keys)
        )
    )


    return hashlib.sha1(
        stable_cluster_string.encode(
            "utf-8"
        )
    ).hexdigest()[:12]


# ============================================================
# 6. CREATE CANONICAL ENTITY REGISTRY
# ============================================================

canonical_rows = []

mention_to_canonical = {}


for root, member_keys in clusters.items():

    member_records = [
        mention_lookup[key]
        for key in member_keys
    ]


    # --------------------------------------------------------
    # Cluster integrity
    # --------------------------------------------------------

    instances = sorted({
        str(record["instance_id"])
        for record in member_records
    })

    entity_types = sorted({
        str(record["entity_type"])
        for record in member_records
    })


    if len(instances) != 1:

        raise ValueError(
            f"Cross-instance cluster detected: "
            f"{root} -> {instances}"
        )


    if len(entity_types) != 1:

        raise ValueError(
            f"Cross-type cluster detected: "
            f"{root} -> {entity_types}"
        )


    instance_id = instances[0]

    entity_type = (
        entity_types[0]
        .upper()
    )


    # --------------------------------------------------------
    # Canonical name
    # --------------------------------------------------------

    canonical_name = (
        choose_canonical_name(
            member_records
        )
    )


    # --------------------------------------------------------
    # Human-readable slug
    # --------------------------------------------------------

    slug = canonical_slug(
        canonical_name
    )


    # --------------------------------------------------------
    # UNIQUE CLUSTER FINGERPRINT
    # --------------------------------------------------------

    fingerprint = (
        cluster_fingerprint(
            member_keys
        )
    )


    # --------------------------------------------------------
    # Stable canonical ID
    # --------------------------------------------------------

    canonical_id = (
        f"{entity_type}"
        f"::{instance_id}"
        f"::{slug}"
        f"::{fingerprint}"
    )


    # --------------------------------------------------------
    # Aliases
    # --------------------------------------------------------

    aliases = sorted({
        str(record["name"]).strip()
        for record in member_records
        if str(record["name"]).strip()
    })


    # --------------------------------------------------------
    # Sources
    # --------------------------------------------------------

    source_types = sorted({
        str(record.get("source_type", ""))
        for record in member_records
        if str(record.get("source_type", ""))
    })


    source_ids = sorted({
        str(record.get("source_id", ""))
        for record in member_records
        if str(record.get("source_id", ""))
    })


    # --------------------------------------------------------
    # Registry row
    # --------------------------------------------------------

    canonical_rows.append({

        "canonical_id":
            canonical_id,

        "instance_id":
            instance_id,

        "entity_type":
            entity_type,

        "canonical_name":
            canonical_name,

        "aliases":
            aliases,

        "mention_count":
            len(member_keys),

        "source_types":
            source_types,

        "source_ids":
            source_ids,

        "cluster_fingerprint":
            fingerprint,

        "cluster_root":
            root,

    })


    # --------------------------------------------------------
    # Mention -> canonical mapping
    # --------------------------------------------------------

    for mention_key in member_keys:

        mention_to_canonical[
            mention_key
        ] = canonical_id


# ============================================================
# 7. CANONICAL ENTITY DATAFRAME
# ============================================================

canonical_entities = pd.DataFrame(
    canonical_rows
)


# ============================================================
# 8. ATTACH IDs TO MENTION TABLE
# ============================================================

mentions[
    "canonical_id"
] = mentions[
    "mention_key"
].map(
    mention_to_canonical
)


# Attach canonical names as well.
canonical_name_lookup = dict(
    zip(
        canonical_entities[
            "canonical_id"
        ],
        canonical_entities[
            "canonical_name"
        ]
    )
)


mentions[
    "canonical_name"
] = mentions[
    "canonical_id"
].map(
    canonical_name_lookup
)


# ============================================================
# 9. HARD INTEGRITY CHECKS
# ============================================================

uf_cluster_count = len(
    clusters
)

canonical_row_count = len(
    canonical_entities
)

unique_canonical_ids = (
    canonical_entities[
        "canonical_id"
    ]
    .nunique()
)

mention_unique_ids = (
    mentions[
        "canonical_id"
    ]
    .nunique()
)


print()
print("=" * 70)
print("CANONICAL ID INTEGRITY CHECK")
print("=" * 70)

print(
    "Union-Find clusters       :",
    uf_cluster_count
)

print(
    "Canonical entity rows     :",
    canonical_row_count
)

print(
    "Unique canonical IDs      :",
    unique_canonical_ids
)

print(
    "Mention-table unique IDs  :",
    mention_unique_ids
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    uf_cluster_count
    == canonical_row_count
), (
    "Canonical entity row count does not "
    "match Union-Find cluster count."
)


assert (
    canonical_row_count
    == unique_canonical_ids
), (
    "Canonical ID collision detected."
)


assert (
    mention_unique_ids
    == uf_cluster_count
), (
    "Mention mapping does not cover all "
    "canonical clusters correctly."
)


assert (
    mentions["canonical_id"]
    .notna()
    .all()
), (
    "Some mentions have no canonical ID."
)


print()
print(
    "✓ PASS: Exactly one unique canonical ID "
    "per Union-Find cluster."
)


# ============================================================
# 10. CHECK CROSS-INSTANCE / CROSS-TYPE AGAIN
# ============================================================

print()

print(
    "Cross-instance canonical entities:",
    sum(
        canonical_entities[
            "instance_id"
        ].isna()
    )
)

print(
    "Cross-type canonical entities:",
    0
)


# ============================================================
# 11. SAVE OUTPUTS
# ============================================================

CANONICAL_ENTITIES_PATH = os.path.join(
    OUTPUT_DIR,
    "canonical_entities.jsonl"
)

RESOLUTION_MAP_PATH = os.path.join(
    OUTPUT_DIR,
    "entity_resolution_map.jsonl"
)


# ------------------------------------------------------------
# Canonical entity registry
# ------------------------------------------------------------

canonical_entities.to_json(
    CANONICAL_ENTITIES_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# ------------------------------------------------------------
# Mention -> canonical ID map
# ------------------------------------------------------------

resolution_map = mentions[
    [
        "instance_id",
        "source_type",
        "source_id",
        "local_id",
        "mention_key",
        "entity_type",
        "name",
        "canonical_name",
        "canonical_id",
    ]
].copy()


resolution_map.to_json(
    RESOLUTION_MAP_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


print()
print(
    "Saved canonical entities:"
)

print(
    CANONICAL_ENTITIES_PATH
)


print()

print(
    "Saved entity resolution map:"
)

print(
    RESOLUTION_MAP_PATH
)


# ============================================================
# 12. SUMMARY
# ============================================================

print()

print("=" * 70)
print("STEP 14 COMPLETE")
print("=" * 70)

print(
    "Mentions           :",
    f"{len(mentions):,}"
)

print(
    "Canonical entities :",
    f"{len(canonical_entities):,}"
)

print(
    "Unique IDs         :",
    f"{canonical_entities['canonical_id'].nunique():,}"
)


# ============================================================
# 13. DISPLAY SAMPLE
# ============================================================

display(
    canonical_entities[
        [
            "instance_id",
            "entity_type",
            "canonical_name",
            "canonical_id",
            "mention_count",
            "aliases",
        ]
    ]
    .head(30)
)


# ============================================================
# CLEANUP
# ============================================================

del canonical_rows

gc.collect()

Current Union-Find clusters: 40325

CANONICAL ID INTEGRITY CHECK
Union-Find clusters       : 40325
Canonical entity rows     : 40325
Unique canonical IDs      : 40325
Mention-table unique IDs  : 40325

✓ PASS: Exactly one unique canonical ID per Union-Find cluster.

Cross-instance canonical entities: 0
Cross-type canonical entities: 0

Saved canonical entities:
/content/drive/MyDrive/GC1/entity_resolution_output/canonical_entities.jsonl

Saved entity resolution map:
/content/drive/MyDrive/GC1/entity_resolution_output/entity_resolution_map.jsonl

STEP 14 COMPLETE
Mentions           : 61,392
Canonical entities : 40,325
Unique IDs         : 40,325


,instance_id,entity_type,canonical_name,canonical_id,mention_count,aliases
0,JBEAP-I01,PROJECT,JBEAP,PROJECT::JBEAP-I01::jbeap::70b147b02096,134,[JBEAP]
1,JBEAP-I01,ISSUE,JBEAP-7698,ISSUE::JBEAP-I01::jbeap-7698::fcc8a4360aa0,1,[JBEAP-7698]
2,JBEAP-I01,PERSON,<<|author_displayName|668abc82-6cf3-4b94-9217-...,PERSON::JBEAP-I01::author-displayname-668abc82...,10,[<<|author_displayName|668abc82-6cf3-4b94-9217...
3,JBEAP-I01,PERSON,<<|author_displayName|4cba0b88-c353-4847-8bea-...,PERSON::JBEAP-I01::author-displayname-4cba0b88...,2,[<<|author_displayName|4cba0b88-c353-4847-8bea...
4,JBEAP-I01,PERSON,<<|author_displayName|b1413bac-20b5-4688-9f18-...,PERSON::JBEAP-I01::author-displayname-b1413bac...,2,[<<|author_displayName|b1413bac-20b5-4688-9f18...
5,JBEAP-I01,COMPONENT,Documentation,COMPONENT::JBEAP-I01::documentation::33c24fc0f577,47,[Documentation]
6,JBEAP-I01,ISSUE,JBEAP-6030,ISSUE::JBEAP-I01::jbeap-6030::ae2302f10b96,1,[JBEAP-6030]
7,JBEAP-I01,ISSUE,JBEAP-8472,ISSUE::JBEAP-I01::jbeap-8472::1daf2abdf604,1,[JBEAP-8472]
8,JBEAP-I01,ISSUE,JBEAP-11399,ISSUE::JBEAP-I01::jbeap-11399::32f179f744df,1,[JBEAP-11399]
9,JBEAP-I01,ISSUE,JBEAP-11410,ISSUE::JBEAP-I01::jbeap-11410::dcf95bba76c6,1,[JBEAP-11410]


0

In [21]:
# ============================================================
# STEP 15: EXPORT CANONICAL ENTITY REGISTRY + RESOLUTION MAP
# ============================================================
def write_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')

resolution_map_rows = []
for r in mentions.to_dict('records'):
    resolution_map_rows.append({
        'repository_project': r['repository_project'],
        'project_key': r['project_key'],
        'instance_id': r['instance_id'],
        'source_type': r['source_type'],
        'source_id': r['source_id'],
        'local_id': r['local_id'],
        'mention_key': r['mention_key'],
        'entity_type': r['entity_type'],
        'original_name': r['name'],
        'normalized_name': r['normalized_name'],
        'canonical_id': r['canonical_id'],
        'canonical_name': r['canonical_name'],
    })

canonical_rows = []
for canonical_id, group in mentions.groupby('canonical_id'):
    methods = []
    roots = set(group['mention_key'])
    for e in resolution_edges:
        if e['mention_a'] in roots and e['mention_b'] in roots:
            methods.append(e['method'])
    canonical_rows.append({
        'canonical_id': canonical_id,
        'entity_type': group['entity_type'].iloc[0],
        'canonical_name': group['canonical_name'].iloc[0],
        'instance_id': group['instance_id'].iloc[0],
        'project_key': group['project_key'].iloc[0],
        'aliases': sorted(set(group['name'])),
        'normalized_aliases': sorted(set(group['normalized_name'])),
        'explicit_ids': sorted(set(group['explicit_id'].dropna().astype(str))),
        'resolution_methods': sorted(set(methods)) or ['SINGLETON'],
        'mention_count': len(group),
        'mentions': [
            {'source_type': x.source_type, 'source_id': x.source_id, 'local_id': x.local_id}
            for x in group.itertuples()
        ]
    })

canonical_path = OUTPUT_ROOT / 'canonical_entities.jsonl'
map_path = OUTPUT_ROOT / 'entity_resolution_map.jsonl'
write_jsonl(canonical_rows, canonical_path)
write_jsonl(resolution_map_rows, map_path)
print('Saved:', canonical_path)
print('Saved:', map_path)

Saved: /content/drive/MyDrive/GC1/entity_resolution_output/canonical_entities.jsonl
Saved: /content/drive/MyDrive/GC1/entity_resolution_output/entity_resolution_map.jsonl


In [22]:
# ============================================================
# STEP 16: REWRITE LOCAL ENTITY REFERENCES TO CANONICAL IDs
# ============================================================
# Local IDs are source-scoped, so lookup always includes instance/source/source_id.
map_lookup = {
    (r['instance_id'], r['source_type'], r['source_id'], r['local_id']): r['canonical_id']
    for r in resolution_map_rows
}

# Only rewrite values that exactly equal a local entity ID from THIS source record.
# This safely handles relationship/event/dependency/blocker structures without
# guessing that arbitrary person names or free text are entity references.
def rewrite_local_refs(obj, local_to_canonical):
    if isinstance(obj, dict):
        return {k: rewrite_local_refs(v, local_to_canonical) for k, v in obj.items()}
    if isinstance(obj, list):
        return [rewrite_local_refs(v, local_to_canonical) for v in obj]
    if isinstance(obj, str) and obj in local_to_canonical:
        return local_to_canonical[obj]
    return obj

resolved_records = []
for record in all_records:
    out = copy.deepcopy({k:v for k,v in record.items() if not k.startswith('_')})
    src = out['source']
    local_to_canonical = {}
    for entity in out['entities']:
        key = (out['instance_id'], src['source_type'], src['source_id'], entity['local_id'])
        local_to_canonical[entity['local_id']] = map_lookup[key]
        entity['canonical_id'] = map_lookup[key]
        entity['normalized_name'] = normalize_name(entity['name'])
    for section in ['relationships','events','decisions','action_items','blockers','dependencies']:
        out[section] = rewrite_local_refs(out.get(section, []), local_to_canonical)
    resolved_records.append(out)

resolved_path = OUTPUT_ROOT / 'resolved_extraction_records.jsonl'
write_jsonl(resolved_records, resolved_path)
print('Saved:', resolved_path)

Saved: /content/drive/MyDrive/GC1/entity_resolution_output/resolved_extraction_records.jsonl


In [23]:
# ============================================================
# STEP 17: INTEGRITY CHECKS
# ============================================================
errors = []
canonical_ids = {r['canonical_id'] for r in canonical_rows}

# Every mention maps exactly once.
if len(resolution_map_rows) != len(mentions):
    errors.append('Resolution map row count does not equal mention count.')
if len({r['mention_key'] for r in resolution_map_rows}) != len(resolution_map_rows):
    errors.append('Duplicate mention_key in resolution map.')
if not {r['canonical_id'] for r in resolution_map_rows}.issubset(canonical_ids):
    errors.append('Resolution map contains unknown canonical IDs.')

# No canonical cluster may cross project instances or entity types.
for cid, group in mentions.groupby('canonical_id'):
    if group['instance_id'].nunique() != 1:
        errors.append(f'{cid}: cluster crosses instances')
    if group['entity_type'].nunique() != 1:
        errors.append(f'{cid}: cluster crosses entity types')

# Relationship endpoints that were local IDs should now be canonical IDs.
local_id_pattern = re.compile(r'^(JIRA|PRD|MEETING)_.+_E\d+$')
def find_leftover_local_ids(obj, path='root'):
    found = []
    if isinstance(obj, dict):
        for k,v in obj.items(): found += find_leftover_local_ids(v, f'{path}.{k}')
    elif isinstance(obj, list):
        for i,v in enumerate(obj): found += find_leftover_local_ids(v, f'{path}[{i}]')
    elif isinstance(obj, str) and local_id_pattern.match(obj):
        found.append((path,obj))
    return found

for i, r in enumerate(resolved_records):
    for section in ['relationships','events','decisions','action_items','blockers','dependencies']:
        leftovers = find_leftover_local_ids(r.get(section, []), f'record[{i}].{section}')
        errors.extend([f'Unresolved local ID {v} at {p}' for p,v in leftovers])

print('Integrity errors:', len(errors))
for e in errors[:50]: print(' -', e)
if errors:
    raise ValueError('Integrity checks failed. Do not import into Neo4j yet.')
print('SUCCESS: entity-resolution integrity checks passed.')

Integrity errors: 0
SUCCESS: entity-resolution integrity checks passed.


In [24]:
# ============================================================
# STEP 18: EXPORT NEO4J-READY NODE + EDGE CSVs
# ============================================================
# These CSVs are portable. Any teammate can import them into a local Neo4j instance.
neo4j_dir = OUTPUT_ROOT / 'neo4j_import'
neo4j_dir.mkdir(exist_ok=True)

# Canonical entity nodes
node_rows = []
for e in canonical_rows:
    node_rows.append({
        'canonical_id:ID': e['canonical_id'],
        'name': e['canonical_name'],
        'entity_type': e['entity_type'],
        'instance_id': e['instance_id'],
        'project_key': e['project_key'],
        'aliases_json': json.dumps(e['aliases'], ensure_ascii=False),
        ':LABEL': e['entity_type'].title().replace('_','')
    })
nodes_df = pd.DataFrame(node_rows)
nodes_df.to_csv(neo4j_dir / 'entities.csv', index=False)

# Typed relationships from resolved extraction records.
edge_rows = []
for r in resolved_records:
    srcmeta = r['source']
    for rel in r.get('relationships', []):
        a, b = rel.get('source_entity'), rel.get('target_entity')
        if a in canonical_ids and b in canonical_ids:
            edge_rows.append({
                ':START_ID': a,
                ':END_ID': b,
                ':TYPE': rel.get('type') or 'RELATED_TO',
                'timestamp': rel.get('timestamp'),
                'instance_id': r['instance_id'],
                'source_type': srcmeta.get('source_type'),
                'source_id': srcmeta.get('source_id'),
                'confidence': rel.get('confidence'),
                'attributes_json': json.dumps(rel.get('attributes') or {}, ensure_ascii=False)
            })
edges_df = pd.DataFrame(edge_rows)
edges_df.to_csv(neo4j_dir / 'relationships.csv', index=False)

# Evidence/source records remain database-neutral and can later become :Evidence nodes.
evidence_rows = []
for r in resolved_records:
    p = r.get('provenance') or {}
    evidence_rows.append({
        'instance_id': r['instance_id'],
        'source_type': p.get('source_type'),
        'source_id': p.get('source_id'),
        'chunk_id': p.get('chunk_id'),
        'evidence_text': p.get('evidence_text'),
        'file_name': p.get('file_name'),
        'source_timestamp': r.get('source',{}).get('source_timestamp')
    })
pd.DataFrame(evidence_rows).to_csv(neo4j_dir / 'evidence.csv', index=False)

print('Neo4j-ready directory:', neo4j_dir)
print('Entity nodes :', len(nodes_df))
print('Relationships:', len(edges_df))

Neo4j-ready directory: /content/drive/MyDrive/GC1/entity_resolution_output/neo4j_import
Entity nodes : 40325
Relationships: 63305


## What is portable vs. Neo4j-specific?

The files in `entity_resolution_output/` are the canonical shared dataset. They do not require Neo4j to exist. The `neo4j_import/` CSVs are a convenient representation for graph import.

A teammate can copy/sync this output folder, start a fresh local Neo4j database, create constraints, and import the same nodes/relationships. This gives each teammate an independent database containing the same logical graph. Keep the canonical resolved JSONL/CSV outputs under a shared/versioned location so Neo4j can always be rebuilt.

### Important next-stage note
This notebook exports the canonical entity nodes and extracted typed relationships. For the **Temporal KG**, the next notebook should additionally materialize `Event`, `Decision`, `Blocker`, `ActionItem`, and `Evidence` as first-class Neo4j nodes where appropriate, preserve their timestamps/provenance, create constraints/indexes, and validate point-in-time queries `KG(t)`.